[Back to NLP guideline](Natural-Language-Processing.html)


## **Baselines, Benchmarks, and State of the Art in NLP** {#baselines-benchmarks-and-state-of-the-art-in-nlp}

An NLP result is meaningful only when it is compared against an appropriate reference system under a shared evaluation protocol. A baseline establishes the starting point, a benchmark defines the environment in which systems are compared, and a state-of-the-art claim describes the strongest reported result under a particular setting.

This chapter connects these three ideas as a research workflow. It focuses on how to construct informative baselines, understand what a benchmark actually standardizes, and decide whether an apparent improvement is fair, reproducible, efficient, and practically meaningful.

### **From Building Models to Comparing Models** {#from-building-models-to-comparing-models}

Training a model and obtaining a score does not by itself demonstrate that the model is useful. Suppose a sentiment classifier reaches $89\%$ accuracy. That number may appear strong, but if $90\%$ of the test examples belong to the positive class, a system that always predicts *positive* reaches $90\%$ without reading the text. The learned model is then worse than a trivial baseline.

Model comparison begins by holding the experimental conditions fixed:

```text
Same task definition
    -> Same data and split
    -> Same preprocessing
    -> Same evaluation metric
    -> Baseline result
    -> Candidate result
    -> Performance and cost comparison
```

Let $M_b$ be the evaluation score of a baseline and $M_c$ the score of a candidate model. The absolute improvement is:

$$
\Delta M = M_c - M_b
$$

$\Delta M$ tells us how many metric points the candidate gained under the shared protocol. If baseline macro-F1 is $0.72$ and candidate macro-F1 is $0.76$, then $\Delta M=0.04$. This difference is only the beginning of the analysis: we must still ask whether it is stable across random seeds, whether the systems use the same data, and whether four points justify additional latency, memory, annotation, or training cost.

A useful comparison does not rely on one baseline. It builds a **baseline ladder** in which each level tests a different hypothesis:

| Baseline level | Information used | Question it answers |
|---|---|---|
| Majority or random | Label distribution only | Does the system beat a prediction that ignores text? |
| Heuristic or lexicon | Human-written lexical rules | Can obvious linguistic cues solve much of the task? |
| Sparse linear model | Word and n-gram statistics | Are surface patterns already sufficient? |
| Lightweight neural model | Learned embeddings and word order | Does learned contextual structure add value? |
| Pretrained model | Knowledge transferred from large corpora | How much does pretraining improve the task? |
| Prompt-based model | Instructions and demonstrations | Can a general model solve the task without task-specific training? |

The ladder turns model development into diagnosis. If a complex Transformer barely beats TF-IDF with logistic regression, the task may be dominated by vocabulary cues, the dataset may be too small, or the evaluation may not reward deeper understanding. If a keyword rule performs unusually well, the data may contain annotation shortcuts. A baseline is therefore not merely a weak competitor; it is an instrument for learning what the task and dataset actually require.

### **Baselines in NLP** {#baselines-in-nlp}

A **baseline** is a simple, explicit, reproducible reference system against which a proposed method is evaluated. Simple does not necessarily mean poor. TF-IDF with a linear classifier can be a strong baseline for document classification, BM25 can remain difficult to beat in lexical retrieval, and an extractive heuristic can be competitive for some summarization datasets.

Baselines serve several roles at once:

- **Lower bound:** establish performance that a meaningful system should exceed.
- **Sanity check:** reveal broken labels, leakage, metric mistakes, or a model that failed to learn.
- **Diagnostic control:** test whether lexical cues, label priors, word order, or pretraining explain the improvement.
- **Research reference:** make results comparable with previous and future work.
- **Engineering reference:** show whether additional complexity is worth its operational cost.

A baseline should be selected before inspecting final test results. Repeatedly inventing a new baseline after seeing the test score turns the test set into part of model development and weakens the credibility of the comparison.

#### **What Makes a Good Baseline?** {#what-makes-a-good-baseline}

A good baseline is **relevant** to the task, **simple enough to understand**, **strong enough to be informative**, and **cheap enough to reproduce**. It should expose what information it uses and what information it deliberately ignores.

| Property | Why it matters | Warning sign |
|---|---|---|
| Task appropriate | The output structure matches the real problem | Using majority-class prediction as the only NER baseline |
| Reproducible | Others can recreate the score | Missing split, seed, preprocessing, or model version |
| Interpretable | Its behavior can diagnose the dataset | A large opaque model described only as "the baseline" |
| Competitive | Improvement over it is meaningful | Comparing only with an intentionally weak method |
| Resource aware | Cost differences remain visible | Reporting accuracy while hiding a 100x compute increase |
| Stable | Results do not depend on one lucky run | Reporting the best seed without mean and variation |

The choice of baseline should correspond to a claim. Consider three candidate claims:

| Research claim | Necessary comparison |
|---|---|
| "The model learns from text" | Data-independent baseline |
| "Context modeling helps" | Order-insensitive model such as TF-IDF or mean embeddings |
| "Pretraining helps" | Same downstream architecture trained from scratch or with a frozen encoder |

This idea is close to a controlled experiment. A baseline should remove the mechanism being claimed while keeping the rest of the setting as comparable as possible.

Some related terms should not be confused:

- A **baseline** is a reference system that can make predictions on the task.
- An **ablation** removes or changes one component of the proposed model to identify its contribution.
- A **control condition** isolates a possible alternative explanation, such as shuffled labels or randomized features.
- An **oracle** uses information that would not normally be available at inference time and estimates an optimistic upper bound.

For example, an oracle retrieval system that always places the gold passage first is useful for measuring the maximum downstream QA performance, but it is not a deployable baseline because it uses the answer key.

#### **Data-Independent Baselines** {#data-independent-baselines}

Data-independent baselines ignore the input text $x$. Their predictions may use the label distribution observed in the training set, but words, syntax, and semantics play no role. They are the first defense against misleading results caused by class imbalance.

Assume a classification problem has $K$ classes and training class proportions $p_1,\ldots,p_K$, where $p_k$ is the proportion of examples with class $k$ and $\sum_{k=1}^{K}p_k=1$.

The most common strategies are:

| Strategy | Prediction rule | Expected behavior |
|---|---|---|
| Majority class | Always predict $\arg\max_k p_k$ | Accuracy approaches $\max_k p_k$ |
| Uniform random | Sample each class with probability $1/K$ | Expected accuracy is $1/K$ |
| Stratified random | Sample class $k$ with probability $p_k$ | Expected accuracy is $\sum_k p_k^2$ |
| Constant class | Always predict a chosen class | Tests a specific class or metric edge case |

For a dataset with class proportions $0.70$, $0.20$, and $0.10$:

$$
\text{Majority accuracy}=\max(0.70,0.20,0.10)=0.70
$$

$$
\text{Uniform expected accuracy}=\frac{1}{3}\approx0.333
$$

$$
\text{Stratified expected accuracy}=0.70^2+0.20^2+0.10^2=0.54
$$

The stratified value is the probability that an independently sampled prediction and true label fall in the same class. The majority strategy achieves higher accuracy, but it has zero recall for both minority classes. This is why the baseline must be evaluated with the same metrics that matter for the task. Macro-F1 will expose behavior hidden by aggregate accuracy.

Scikit-learn provides these strategies through `DummyClassifier`, whose predictions intentionally ignore input features. The official behavior of `most_frequent`, `prior`, `stratified`, `uniform`, and `constant` is documented in the [DummyClassifier reference](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html).

<details>
<summary>Python: Comparing data-independent classification baselines</summary>

```python
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

# The text is passed to the API, but every dummy strategy ignores it.
X_train = [f"training document {i}" for i in range(10)]
y_train = ["positive"] * 7 + ["neutral"] * 2 + ["negative"]

X_test = [f"test document {i}" for i in range(10)]
y_test = ["positive"] * 7 + ["neutral"] * 2 + ["negative"]

strategies = {
    "majority": DummyClassifier(strategy="most_frequent"),
    "uniform": DummyClassifier(strategy="uniform", random_state=42),
    "stratified": DummyClassifier(strategy="stratified", random_state=42),
}

for name, baseline in strategies.items():
    # Step 1: fit learns only the class distribution from y_train.
    baseline.fit(X_train, y_train)

    # Step 2: predict without examining any text content.
    predictions = baseline.predict(X_test)

    # Step 3: report both accuracy and macro-F1.
    # Macro-F1 gives every class equal importance.
    accuracy = accuracy_score(y_test, predictions)
    macro_f1 = f1_score(
        y_test,
        predictions,
        average="macro",
        zero_division=0,
    )

    print(f"{name:10s} accuracy={accuracy:.3f} macro_f1={macro_f1:.3f}")
```
</details>

If a text model cannot reliably exceed these baselines, the next step is not automatically to increase model size. First inspect the labels, data split, optimization, and evaluation pipeline. The model may not be learning, or the input text may not contain enough information to predict the target.

#### **Heuristic and Linguistic Baselines** {#heuristic-and-linguistic-baselines}

A heuristic baseline reads the input but does not learn its decision rule from labeled examples. It uses manually specified knowledge such as keywords, sentiment lexicons, regular expressions, morphology, capitalization, gazetteers, or syntactic patterns.

For lexicon sentiment analysis, assign each recognized token $w_i$ a polarity value $\ell(w_i)$. A simple document score is:

$$
s(x)=\sum_{i=1}^{n}\ell(w_i)
$$

$x=(w_1,\ldots,w_n)$ is the tokenized text, $n$ is its number of tokens, and $\ell(w_i)$ may be positive, negative, or zero. The prediction can use thresholds:

$$
\hat y=
\begin{cases}
\text{positive}, & s(x)>0 \\
\text{neutral}, & s(x)=0 \\
\text{negative}, & s(x)<0
\end{cases}
$$

The score is interpretable because each word contributes visibly. Its weakness is equally clear: language meaning is not a simple sum. Negation can reverse polarity, intensifiers can change magnitude, word senses vary by domain, and sarcasm may contradict literal wording.

Compare these examples:

| Text | Naive lexical reading | Linguistic issue |
|---|---|---|
| `The service was excellent.` | Positive | Straightforward lexical cue |
| `The service was not excellent.` | Positive unless negation is handled | Negation scope |
| `The battery is sick.` | Negative in general English | Domain and slang meaning |
| `Great, it crashed again.` | Positive lexical item | Sarcasm and discourse context |

The following baseline adds a small negation window. It remains deliberately limited, but the code makes the hypothesis explicit and creates useful failure cases for later models.

<details>
<summary>Python: Lexicon sentiment baseline with simple negation handling</summary>

```python
import re

LEXICON = {
    "excellent": 2,
    "great": 2,
    "good": 1,
    "helpful": 1,
    "bad": -1,
    "terrible": -2,
    "crashed": -2,
    "slow": -1,
}
NEGATIONS = {"not", "never", "no", "hardly"}

def tokenize(text):
    # Keep a transparent tokenizer so every rule is inspectable.
    return re.findall(r"[a-z']+", text.lower())

def lexicon_sentiment(text, negation_window=3):
    tokens = tokenize(text)
    score = 0
    remaining_negation_scope = 0

    for token in tokens:
        # Step 1: open a short negation scope after a negation word.
        if token in NEGATIONS:
            remaining_negation_scope = negation_window
            continue

        # Step 2: look up the token's manually assigned polarity.
        contribution = LEXICON.get(token, 0)

        # Step 3: reverse polarity while the token is in negation scope.
        if contribution != 0 and remaining_negation_scope > 0:
            contribution *= -1

        score += contribution
        remaining_negation_scope = max(0, remaining_negation_scope - 1)

    # Step 4: convert the transparent numeric score into a label.
    if score > 0:
        return "positive", score
    if score < 0:
        return "negative", score
    return "neutral", score

examples = [
    "The service was excellent.",
    "The service was not excellent.",
    "Great, it crashed again.",  # The rule still misses sarcasm.
]

for text in examples:
    print(text, "->", lexicon_sentiment(text))
```
</details>

Heuristic baselines are particularly valuable when labeled data is scarce, regulations require traceable decisions, or domain experts already possess high-precision rules. They are also useful for **weak supervision**, where rules produce noisy labels for a larger unlabeled corpus.

Their main limitation is maintenance. Each new domain, language variety, and failure pattern may require another rule. A heuristic baseline should therefore report both aggregate performance and coverage: a high-precision rule that fires on only $5\%$ of examples is not directly comparable with a classifier that predicts every example unless abstention is part of the task.

#### **Classical Machine Learning Baselines** {#classical-machine-learning-baselines}

Classical NLP baselines learn from labeled data but keep the representation and decision function relatively simple. A standard pipeline is:

```text
Raw text
    -> token and n-gram counts
    -> TF-IDF weighting
    -> linear classifier
    -> label probabilities or decision scores
```

This pipeline is often stronger than its simplicity suggests. Text vectors are high-dimensional and sparse, and many classification tasks contain highly predictive words or short phrases. A linear model can assign one weight to each feature and combine thousands of weak lexical signals efficiently.

For a multi-class linear classifier, class $k$ receives a score:

$$
z_k=w_k^Tx+b_k
$$

$x$ is the TF-IDF document vector, $w_k$ contains learned feature weights for class $k$, and $b_k$ is a bias term. Logistic regression converts these scores into probabilities with softmax:

$$
P(y=k\mid x)=\frac{e^{z_k}}{\sum_{j=1}^{K}e^{z_j}}
$$

The exponential makes every value positive, and the denominator normalizes all $K$ class values so they sum to one. A feature with a large positive weight for class $k$ increases that class score; a large negative weight decreases it.

Common classical baselines differ in their assumptions:

| Model | Main strength | Typical limitation |
|---|---|---|
| Multinomial Naive Bayes | Very fast and effective with word counts | Conditional-independence assumption and less flexible decision boundary |
| Logistic Regression | Probabilistic output and interpretable weights | Context limited to supplied features |
| Linear SVM | Strong margin-based classifier for sparse text | Scores are not probabilities without calibration |
| Character n-gram linear model | Robust to spelling variation and morphology | May exploit orthographic shortcuts |

The feature representation is explained in [Text Representation](03-text-representation.html). Here the important point is experimental: this baseline determines whether a new model improves over surface lexical statistics.

<details>
<summary>Python: TF-IDF and logistic regression baseline with inspection</summary>

```python
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline

texts = [
    "excellent acting and a moving story",
    "a thoughtful and beautifully written film",
    "the performances were warm and convincing",
    "smart dialogue with a satisfying ending",
    "I loved the pacing and the characters",
    "a delightful film with excellent music",
    "the plot was engaging and original",
    "good acting carried the entire movie",
    "a boring story with terrible dialogue",
    "the film was slow and painfully predictable",
    "bad acting and a disappointing ending",
    "I hated the flat characters",
    "the plot was confusing and unoriginal",
    "a dull movie with forgettable music",
    "terrible pacing ruined the experience",
    "the story was weak and badly written",
]
labels = np.array(["positive"] * 8 + ["negative"] * 8)

# Step 1: put representation and classifier in one reproducible pipeline.
baseline = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),  # Learn words and short phrases such as "not good".
        sublinear_tf=True,   # Reduce the dominance of repeated terms.
    )),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])

# Step 2: use the same folds for every future candidate model comparison.
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)
scores = cross_validate(
    baseline,
    texts,
    labels,
    cv=cv,
    scoring={"accuracy": "accuracy", "macro_f1": "f1_macro"},
)

print("Mean accuracy:", scores["test_accuracy"].mean())
print("Mean macro-F1:", scores["test_macro_f1"].mean())

# Step 3: fit once on all available training data for feature inspection.
baseline.fit(texts, labels)
vectorizer = baseline.named_steps["tfidf"]
classifier = baseline.named_steps["classifier"]

feature_names = vectorizer.get_feature_names_out()
weights = classifier.coef_[0]

print("Most positive features:", feature_names[np.argsort(weights)[-5:]])
print("Most negative features:", feature_names[np.argsort(weights)[:5]])
```
</details>

Feature inspection is part of the baseline's diagnostic value. If a model predicts sentiment mainly from movie titles, usernames, or formatting artifacts, it may perform well for the wrong reason. The same shortcut can later be learned by a Transformer, but it is easier to notice in a transparent linear baseline.

Classical baselines struggle when the task depends on long-range composition, implicit meaning, world knowledge, or contextual word senses. Nevertheless, they should not be skipped. A new neural model that cannot beat a properly tuned sparse linear model has not yet justified its complexity.

#### **Lightweight Neural Baselines** {#lightweight-neural-baselines}

Lightweight neural baselines form a bridge between sparse linear models and large pretrained Transformers. They learn dense representations from the task data and can test whether word order or local context matters without introducing billions of pretraining tokens as an additional source of information.

Three common choices are:

| Baseline | Context mechanism | Diagnostic question |
|---|---|---|
| Mean embedding classifier | Average token embeddings | Do dense lexical representations help without word order? |
| Text CNN | Convolution over local windows | Are local phrases and n-gram-like patterns important? |
| BiLSTM | Recurrent context in both directions | Does sequential context improve the task? |

For a mean embedding baseline, token embeddings $e_1,\ldots,e_n$ are averaged:

$$
h=\frac{1}{n}\sum_{i=1}^{n}e_i
$$

$e_i\in\mathbb{R}^d$ is the $d$-dimensional embedding of token $i$, $n$ is the number of non-padding tokens, and $h$ is a fixed-size sentence representation. A classifier then predicts:

$$
P(y\mid x)=\operatorname{softmax}(Wh+b)
$$

The average is deliberately order-insensitive: *dog bites person* and *person bites dog* contain the same tokens and therefore receive the same representation if their embeddings are identical. This weakness is useful because it creates a clean control. If a BiLSTM substantially outperforms the mean model under the same training conditions, word order and contextual composition are likely contributing.

<details>
<summary>Python: Mask-aware mean embedding classifier</summary>

```python
import torch
import torch.nn as nn

class MeanEmbeddingClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_classes, pad_id=0):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=pad_id,
        )
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, token_ids, attention_mask):
        # Step 1: map token IDs to trainable dense vectors.
        token_vectors = self.embedding(token_ids)

        # Step 2: give real tokens weight 1 and padding tokens weight 0.
        mask = attention_mask.unsqueeze(-1).to(token_vectors.dtype)
        masked_vectors = token_vectors * mask

        # Step 3: average only the non-padding token vectors.
        token_count = mask.sum(dim=1).clamp(min=1.0)
        sentence_vector = masked_vectors.sum(dim=1) / token_count

        # Step 4: produce one score for each class.
        return self.classifier(sentence_vector)

model = MeanEmbeddingClassifier(
    vocab_size=20_000,
    embedding_dim=128,
    num_classes=3,
)

token_ids = torch.tensor([
    [15, 82, 41, 0, 0],
    [91, 37, 12, 66, 5],
])
attention_mask = (token_ids != 0).long()

logits = model(token_ids, attention_mask)
print(logits.shape)  # [batch_size=2, num_classes=3]
```
</details>

CNN and BiLSTM architectures are explained in [Modeling in NLP](04-modeling.html). In a baseline experiment they should remain intentionally small, use the same tokenizer and training split as other candidates, and receive a documented tuning budget. Otherwise, a poorly tuned neural baseline can make a proposed model look stronger than it really is.

Lightweight neural models are not guaranteed to beat sparse linear models. With limited labels and strong lexical cues, TF-IDF may generalize better, train faster, and be easier to debug. The comparison is informative precisely because the outcome is not predetermined.

#### **Pretrained and Prompt-Based Baselines** {#pretrained-and-prompt-based-baselines}

Modern NLP experiments often require a pretrained reference because training only on the target dataset ignores knowledge that is readily available through transfer learning. However, "using a pretrained model" describes several materially different settings.

| Setting | Parameters updated | Task examples required | Main comparison question |
|---|---|---:|---|
| Frozen encoder + linear head | Only task head | Labeled examples | Are pretrained features useful without changing the encoder? |
| Full fine-tuning | Encoder and task head | Labeled examples | What is the standard task-adapted pretrained performance? |
| Zero-shot prompting | No gradient update | No labeled training examples | Can instructions and label descriptions define the task? |
| Few-shot prompting | No gradient update | A few demonstrations in context | How much do demonstrations improve in-context behavior? |

> ![BERT pretraining and downstream fine-tuning](assets/baseline-pretraining-finetuning.png)
>
> Pretraining learns reusable parameters from unlabeled text objectives. Fine-tuning initializes several task-specific systems from the same pretrained backbone and adapts them with labeled data. This makes a pretrained baseline fundamentally different from a model trained only on the downstream dataset.
>
> Source: [BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding, Figure 1](https://arxiv.org/abs/1810.04805)

For a frozen encoder, let $f_{\phi}$ be a pretrained model with fixed parameters $\phi$:

$$
h=f_{\phi}(x), \qquad \hat y=g_{\theta}(h)
$$

$h$ is the contextual representation of input $x$, and only the classifier parameters $\theta$ in $g_{\theta}$ are learned. Full fine-tuning updates both $\phi$ and $\theta$. Comparing these settings helps separate the value of pretrained representations from the value of adapting the entire encoder.

<details>
<summary>Python: Frozen Transformer encoder with logistic regression</summary>

```python
import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from transformers import AutoModel, AutoTokenizer

checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
encoder = AutoModel.from_pretrained(checkpoint)
encoder.eval()

# Freeze the encoder: the downstream classifier is the only learned component.
for parameter in encoder.parameters():
    parameter.requires_grad = False

@torch.no_grad()
def encode_texts(texts, batch_size=16):
    all_vectors = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )

        # Step 1: obtain one contextual vector for every token.
        token_vectors = encoder(**encoded).last_hidden_state

        # Step 2: mean-pool real tokens while excluding padding positions.
        mask = encoded["attention_mask"].unsqueeze(-1)
        pooled = (token_vectors * mask).sum(dim=1) / mask.sum(dim=1)
        all_vectors.append(pooled.cpu().numpy())

    return np.vstack(all_vectors)

# X_train_text and X_test_text must use the same split as other baselines.
train_vectors = encode_texts(X_train_text)
test_vectors = encode_texts(X_test_text)

classifier = LogisticRegression(max_iter=1000, random_state=42)
classifier.fit(train_vectors, y_train)
predictions = classifier.predict(test_vectors)
```
</details>

Prompt-based baselines define labels through natural-language instructions rather than a trained task head. One common zero-shot classifier reformulates each candidate label as a natural-language hypothesis and uses an NLI model's entailment score. Hugging Face documents this behavior in its official [zero-shot classification pipeline](https://huggingface.co/docs/transformers/main_classes/pipelines?highlight=textgeneration#transformers.ZeroShotClassificationPipeline).

<details>
<summary>Python: Zero-shot sentiment baseline with an NLI model</summary>

```python
from transformers import pipeline

# The checkpoint is pretrained and then fine-tuned for natural language inference.
zero_shot = pipeline(
    task="zero-shot-classification",
    model="facebook/bart-large-mnli",
)

text = "The interface is beautiful, but the application crashes every hour."
candidate_labels = ["positive", "neutral", "negative"]

result = zero_shot(
    text,
    candidate_labels=candidate_labels,
    # Label wording is part of the experimental specification.
    hypothesis_template="The overall sentiment of this review is {}.",
)

print(result["labels"])
print(result["scores"])
```
</details>

Prompt-based evaluation introduces new variables: prompt wording, label descriptions, demonstration selection, order, decoding settings, model version, and external API behavior. A zero-shot score is not directly comparable with supervised fine-tuning unless the difference in labeled data and adaptation is made explicit.

Pretrained baselines also carry hidden costs and risks. Pretraining data may overlap with evaluation examples, model sizes may differ greatly, and closed systems may change without preserving an exact checkpoint. A fair report should record the full model identifier, revision or access date, prompting template, precision, hardware, and whether retrieval or external tools were used.

The baseline levels can now be compared directly:

| Level | Learns from task labels? | Uses external pretraining? | Context sensitivity | Typical cost |
|---|---:|---:|---:|---:|
| Majority/random | No | No | None | Minimal |
| Heuristic | No | Human knowledge | Rule-limited | Low |
| TF-IDF + linear model | Yes | No | N-gram features | Low |
| Lightweight neural | Yes | Usually no | Architecture-dependent | Medium |
| Frozen pretrained encoder | Head only | Yes | Strong | Medium to high |
| Full fine-tuning | Yes | Yes | Strong | High |
| Zero/few-shot prompting | No gradient update | Yes | Strong | Model/API dependent |

No row is universally best. The purpose of the ladder is to reveal which additional source of information or complexity creates the observed gain.

#### **Task-Specific Baselines** {#task-specific-baselines}

The correct baseline depends on the output structure. A majority-class classifier is meaningful for document classification but inadequate as the only reference for machine translation or retrieval. Each task needs a baseline that can produce the same type of output and can be measured with the same evaluation protocol as the candidate system.

##### **Text Classification** {#text-classification}

For single-label classification, a defensible minimum ladder is:

```text
Majority class
    -> Keyword or lexicon rule
    -> TF-IDF + logistic regression / linear SVM
    -> Frozen pretrained encoder
    -> Fine-tuned pretrained encoder
```

The majority baseline tests label imbalance, the rule system tests obvious shortcuts, and the linear model tests whether sparse lexical features solve the task. The pretrained baselines then quantify transfer learning.

Multi-label classification requires different naive references because every example can have several labels. A useful data-independent baseline predicts each label according to its training prevalence or always predicts labels above a fixed prevalence threshold. Exact-match accuracy may be extremely harsh, so micro-F1, macro-F1, and per-label recall should be reported according to the task goals.

For hierarchical classification, a flat majority label can ignore parent-child structure. A stronger baseline may predict the most frequent valid path through the label hierarchy. The baseline must obey the same output constraints as the proposed system.

##### **Sequence Labeling** {#sequence-labeling}

Sequence labeling predicts one label per token, often with dependencies between neighboring labels. Appropriate baselines include:

- always predicting the outside label `O`;
- assigning each known token its most frequent training tag;
- applying capitalization, suffix, or gazetteer rules;
- using an HMM or CRF with local features;
- using a small BiLSTM or pretrained token classifier.

A most-frequent-tag baseline can be written as:

$$
\hat y(w)=\arg\max_k \operatorname{count}(w,k)
$$

$\operatorname{count}(w,k)$ is the number of times token $w$ received label $k$ in training. Unseen tokens fall back to the globally most common label or a morphology-based rule.

For named entity recognition, always predicting `O` may produce very high token accuracy because most tokens are not entities, while entity-level precision, recall, and F1 are all zero. The lesson is not that the baseline is useless; it reveals why token accuracy is an inappropriate main metric for the application. Traditional structured models are explained in [Modeling in NLP](04-modeling.html), and span-based evaluation is explained in [Evaluation in NLP](06-evaluation.html).

##### **Language Modeling and Generation** {#language-modeling-and-generation}

Generation tasks require baselines that generate complete outputs rather than class labels. Suitable baselines depend strongly on the application:

| Task | Simple baseline | Stronger reference baseline |
|---|---|---|
| Language modeling | Unigram model | Smoothed n-gram language model |
| Summarization | First sentence or Lead-3 | Extractive sentence-ranking system |
| Machine translation | Copy source or phrase table | Statistical MT or small encoder-decoder |
| Dialogue | Most frequent response | Retrieve a response from similar training context |
| Data-to-text | Fixed template | Template with slot-specific rules |

A **model baseline** and a **decoding baseline** are different. Comparing a Transformer with an n-gram model changes the probability model. Comparing greedy decoding with beam search while holding the Transformer fixed changes only inference. Both comparisons are useful, but they support different claims.

Language-model perplexity is comparable only when tokenization and evaluated tokens are aligned. A word-level n-gram model and a subword Transformer may assign probabilities over different units, so raw perplexity values can be misleading. Generation quality should also include task-specific automatic metrics and human inspection rather than relying on likelihood alone.

##### **Retrieval and Ranking** {#retrieval-and-ranking}

Retrieval systems rank documents for a query. A baseline must therefore produce an ordered list over the same document collection. Common levels are:

```text
Random ranking
    -> Popularity or recency ranking
    -> TF-IDF cosine similarity
    -> BM25
    -> Dense bi-encoder retrieval
    -> Cross-encoder reranking
```

Random ranking checks the metric implementation. Popularity or recency baselines test non-textual shortcuts. TF-IDF and BM25 establish strong lexical retrieval references, while dense retrieval tests semantic matching beyond exact term overlap.

All systems must use the same corpus, query set, relevance judgments, and candidate-pool rules. A dense system evaluated after retrieving candidates with BM25 is not directly comparable with end-to-end BM25 unless the two-stage setup is disclosed. Index construction time, storage, query latency, and reranking depth are also part of the system cost.

The practical baseline choice can be summarized as follows:

| Task property | Minimum baseline | Recommended strong baseline |
|---|---|---|
| Imbalanced classification | Majority and stratified dummy | TF-IDF + class-aware linear model |
| Lexically driven classification | Keyword rule | Word/character n-gram linear model |
| Token-aligned output | `O` or most-frequent tag | CRF or lightweight token classifier |
| Free-form generation | Copy/template/Lead-3 | Small pretrained encoder-decoder |
| Lexical retrieval | Random ranking | BM25 |
| Semantic retrieval | BM25 | Dense bi-encoder plus documented reranker |

The best baseline set is usually small but layered. It should include a trivial lower bound, a strong conventional method, and a reference that isolates the mechanism being claimed. The next section explains how datasets, metrics, splits, and protocols package these comparisons into an NLP benchmark.

### **Benchmarking in NLP** {#benchmarking-in-nlp}

A baseline answers, "What simple or established system should the proposed model beat?" A benchmark answers a broader question: "Under which shared conditions should different systems be measured?" The distinction matters because a model, a metric, and a test dataset do not create a meaningful comparison on their own. A score of 90 can describe accuracy, macro-F1, span F1, BLEU, or a human preference rate; it can come from a public development set, a hidden test set, or a test set that influenced model selection. The number becomes interpretable only when the complete measurement protocol is known.

Benchmarking therefore sits between model development and scientific claims:

$$
\text{models} \longrightarrow \text{shared protocol} \longrightarrow \text{comparable evidence} \longrightarrow \text{claim}
$$

The [Evaluation in NLP](06-evaluation.html) chapter explains how individual metrics diagnose model behavior. This chapter focuses on the larger contract that decides which data, task formulation, adaptation procedure, metrics, and reporting rules every model must share.

#### **What Is an NLP Benchmark?** {#what-is-an-nlp-benchmark}

An NLP benchmark is a standardized and reusable evaluation environment for measuring a defined capability or set of capabilities. It usually includes one or more datasets, but it is not merely a dataset. It also specifies how examples become model inputs, which split is used for development, whether fine-tuning or prompting is allowed, how predictions are converted into answers, which metrics are primary, and how results are reported.

The surrounding terms are related but not interchangeable:

| Term | What it provides | What it does not guarantee |
|---|---|---|
| Dataset | Text, labels, metadata, and often train/dev/test splits | A common model interface, metric, or reporting policy |
| Evaluation | A procedure for measuring predictions against desired behavior | That other researchers use the same procedure |
| Benchmark | A shared task, data, protocol, metrics, and comparison rules | That the measured capability covers real-world language use |
| Leaderboard | A public ordering of submitted benchmark results | Scientific validity, reproducibility, or practical usefulness |

A useful formalization is to treat a benchmark as a specification:

$$
\mathcal{B}=(\mathcal{D},\mathcal{T},\mathcal{S},\mathcal{A},\mathcal{M},\mathcal{R})
$$

where:

- $\mathcal{D}$ is the data and its provenance, annotation policy, license, and version;
- $\mathcal{T}$ is the task definition, including the input-output format and target construct;
- $\mathcal{S}$ is the split policy, such as random, document-level, temporal, domain-held-out, or hidden-test splitting;
- $\mathcal{A}$ is the adaptation protocol, such as full fine-tuning, frozen features, zero-shot prompting, or a fixed number of demonstrations;
- $\mathcal{M}$ is the metric set and any normalization or aggregation rule;
- $\mathcal{R}$ is the reporting contract, including random seeds, confidence intervals, compute, model version, and required subgroup results.

Changing one element can change the scientific question. Fine-tuning BERT on SST-2 measures supervised task adaptation, while prompting a frozen language model on the same sentences measures instruction following under a particular prompt. The dataset is shared, but the benchmark conditions are not.

For a test set with $n$ examples, a benchmark score is commonly an empirical estimate:

$$
\widehat{M}(f)=\frac{1}{n}\sum_{i=1}^{n}m\!\left(f(x_i),y_i\right)
$$

$f$ is the evaluated system, $x_i$ is the $i$-th input, $y_i$ is its gold target, and $m$ is the per-example scoring rule. For accuracy, $m$ equals 1 when the prediction is correct and 0 otherwise. The hat on $\widehat{M}$ is important: the score is an estimate from one finite sample, not the model's permanent ability on every future text. Small or unrepresentative test sets produce uncertain estimates even when the implementation is flawless.

Consider two sentiment systems evaluated on the same 1,000 reviews. Model A reports 89% accuracy after repeatedly selecting prompts on that test set. Model B reports 87% macro-F1 on a hidden test set and includes per-domain results. The larger number does not establish that Model A is better. The models have different metrics, test exposure, and reporting granularity. A benchmark exists to remove these degrees of freedom before results are seen.

A credible benchmark should be:

| Property | Practical question |
|---|---|
| Valid | Does the task actually represent the capability named by the benchmark? |
| Reliable | Would small sampling or annotation changes preserve the main conclusion? |
| Discriminative | Can the test separate meaningfully different systems? |
| Representative | Are domains, users, languages, and difficulty levels relevant to intended use? |
| Reproducible | Can another researcher reconstruct the inputs, protocol, and score? |
| Resistant to gaming | Is it difficult to improve the score without improving the intended capability? |
| Transparent | Are limitations, exclusions, uncertainty, and known contamination documented? |

The final point is conceptual: a benchmark does not discover what "language understanding" means. Its designers operationalize selected parts of that broad idea into observable behavior. Good benchmarking makes this choice explicit rather than treating the resulting score as universal intelligence.

#### **Anatomy of a Benchmark** {#anatomy-of-a-benchmark}

A benchmark begins with a construct: the capability or behavior that should be measured. For named entity recognition, the construct might be reliable extraction of people, organizations, and locations from news text. The benchmark then turns that abstract goal into examples, labels, splits, interfaces, and metrics. Weak benchmarks often begin with an available dataset and only afterward attach a broad capability label to it.

The main components form a measurement chain:

| Component | Design decision | Typical failure if omitted |
|---|---|---|
| Construct | Define the capability and intended population | A narrow task is interpreted as general understanding |
| Task interface | Define inputs, outputs, labels, context, and allowed tools | Models solve different task formulations |
| Data | Record source, time, domain, language, annotation, and license | Hidden artifacts or population bias determine the score |
| Split policy | Separate train/dev/test by the correct unit | Near-duplicate documents leak across splits |
| Adaptation protocol | Fix fine-tuning, prompting, retrieval, and demonstration rules | Extra supervision is mistaken for a better model |
| Metrics | Select primary, diagnostic, subgroup, and cost metrics | One average hides the actual failure mode |
| Aggregation | Specify direction, scale, weights, and missing-task policy | The overall score silently favors some tasks |
| Reporting | Record versions, seeds, uncertainty, compute, and predictions | Results cannot be audited or reproduced |

The split unit deserves special attention in NLP. Randomly splitting sentences is unsafe when several sentences come from the same article, conversation, patient, author, or template. Shared phrasing can let the model recognize the source rather than generalize to new sources. The correct split may need to operate at document, user, time, language, or domain level.

The metric must also match the decision being represented. A medical entity extractor may need high recall for dangerous conditions, while a production retrieval system may care about recall at a fixed latency budget. "Use F1" is not a benchmark design rationale unless the relationship between F1 and the intended use is explained.

The following example turns part of the benchmark contract into executable code. It validates the label space, computes the declared primary metrics, adds a bootstrap confidence interval, and records a fingerprint for the exact test IDs. The fingerprint does not hide the test set; it makes accidental split changes detectable.

<details>
<summary>Python: Encode a classification benchmark as an executable specification</summary>

```python
from dataclasses import asdict, dataclass
import hashlib
import json

import numpy as np
from sklearn.metrics import accuracy_score, f1_score


@dataclass(frozen=True)
class BenchmarkSpec:
    name: str
    version: str
    labels: tuple[str, ...]
    primary_metric: str
    test_split_fingerprint: str
    seed: int = 42


def fingerprint_ids(example_ids):
    """Create a stable identity for the ordered test examples."""
    payload = "\n".join(map(str, example_ids)).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def macro_f1(y_true, y_pred, labels):
    return f1_score(
        y_true,
        y_pred,
        labels=list(labels),
        average="macro",
        zero_division=0,
    )


def bootstrap_interval(y_true, y_pred, metric_fn, seed=42, rounds=2000):
    """Estimate test-sample uncertainty by resampling paired predictions."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    rng = np.random.default_rng(seed)
    scores = []

    # Each round samples test positions with replacement, preserving pairs.
    for _ in range(rounds):
        indices = rng.integers(0, len(y_true), size=len(y_true))
        scores.append(metric_fn(y_true[indices], y_pred[indices]))

    lower, upper = np.percentile(scores, [2.5, 97.5])
    return float(lower), float(upper)


def evaluate_predictions(spec, example_ids, y_true, y_pred):
    # Step 1: reject accidental benchmark or label changes.
    assert fingerprint_ids(example_ids) == spec.test_split_fingerprint
    assert set(y_true).issubset(spec.labels)
    assert set(y_pred).issubset(spec.labels)

    # Step 2: calculate both the primary score and a useful diagnostic score.
    metric_fn = lambda truth, pred: macro_f1(truth, pred, spec.labels)
    score = metric_fn(np.asarray(y_true), np.asarray(y_pred))
    interval = bootstrap_interval(
        y_true,
        y_pred,
        metric_fn,
        seed=spec.seed,
    )

    # Step 3: emit a machine-readable record for later comparison.
    return {
        "benchmark": asdict(spec),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": score,
        "macro_f1_95pct_bootstrap_ci": interval,
        "n_test": len(y_true),
    }


# These IDs and predictions would normally come from the locked test run.
test_ids = ["review-101", "review-102", "review-103", "review-104"]
spec = BenchmarkSpec(
    name="ProductSentiment",
    version="1.0",
    labels=("negative", "neutral", "positive"),
    primary_metric="macro_f1",
    test_split_fingerprint=fingerprint_ids(test_ids),
)

report = evaluate_predictions(
    spec,
    test_ids,
    y_true=["positive", "negative", "neutral", "negative"],
    y_pred=["positive", "neutral", "neutral", "negative"],
)
print(json.dumps(report, indent=2))
```
</details>

Bootstrap intervals answer a narrow but useful question: how much could the score change if a similar test sample had been drawn from the same observed population? They do not correct annotation bias, domain mismatch, contamination, or repeated test-set tuning. Statistical precision and benchmark validity are separate concerns.

An executable evaluator should be accompanied by a benchmark card that documents intended use, excluded uses, collection dates, annotation quality, subgroup coverage, licenses, known leakage, and change history. Code makes the protocol repeatable; documentation makes its meaning inspectable.

#### **Single-Task Benchmarks** {#single-task-benchmarks}

A single-task benchmark concentrates on one input-output relationship, such as sentiment classification, named entity recognition, extractive question answering, machine translation, or next-token prediction. It may contain multiple domains or test sets, but every system is asked to solve the same basic task. This narrow scope makes errors easier to interpret and allows the metric to be tailored to the output structure.

| Benchmark example | Input-output structure | Typical primary metric | What it measures most directly |
|---|---|---|---|
| SST-2 | Sentence $\rightarrow$ sentiment label | Accuracy | Binary sentiment classification on movie-review sentences |
| CoNLL-2003 NER | Token sequence $\rightarrow$ entity spans | Entity-level F1 | Exact entity boundary and type recovery in newswire |
| SQuAD | Passage and question $\rightarrow$ answer span | Exact Match and token F1 | Extractive reading comprehension under the dataset's annotation rules |
| WMT translation task | Source sentence $\rightarrow$ translated sentence | BLEU, chrF, COMET, and/or human judgment | Translation quality for a specified language pair and test year |
| WikiText-103 | Previous tokens $\rightarrow$ next-token probabilities | Perplexity | Predictive language modeling on a particular Wikipedia-derived distribution |

The [SQuAD paper](https://aclanthology.org/D16-1264/) illustrates why the task and scorer must be read together. Suppose the reference answer is `Denver` and a system predicts `Denver, Colorado`. After normalization, Exact Match is 0 because the strings differ. Token precision is $1/2$, token recall is $1$, and token F1 is:

$$
F_1=\frac{2PR}{P+R}=\frac{2(1/2)(1)}{1/2+1}=\frac{2}{3}
$$

$P$ is the fraction of predicted tokens that overlap the reference, and $R$ is the fraction of reference tokens recovered by the prediction. Exact Match rewards exact boundary selection; token F1 gives partial credit. Reporting only one of them changes how the same error is interpreted.

<details>
<summary>Python: A simplified SQuAD-style Exact Match and token-F1 scorer</summary>

```python
from collections import Counter
import re
import string


def normalize_answer(text):
    """Apply the normalization choices that define this scorer."""
    text = text.lower()
    text = "".join(char for char in text if char not in string.punctuation)
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())


def exact_match(prediction, reference):
    return float(normalize_answer(prediction) == normalize_answer(reference))


def token_f1(prediction, reference):
    pred_tokens = normalize_answer(prediction).split()
    ref_tokens = normalize_answer(reference).split()

    # Step 1: handle answers that normalize to an empty sequence.
    if not pred_tokens or not ref_tokens:
        return float(pred_tokens == ref_tokens)

    # Step 2: count overlapping tokens, including repeated occurrences.
    overlap = Counter(pred_tokens) & Counter(ref_tokens)
    common = sum(overlap.values())
    if common == 0:
        return 0.0

    # Step 3: combine partial-answer precision and recall.
    precision = common / len(pred_tokens)
    recall = common / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)


def score_qa_example(prediction, references):
    """Credit the best matching valid human reference."""
    return {
        "exact_match": max(exact_match(prediction, ref) for ref in references),
        "token_f1": max(token_f1(prediction, ref) for ref in references),
    }


example = score_qa_example(
    prediction="Denver, Colorado",
    references=["Denver", "the city of Denver"],
)
print(example)
```
</details>

This implementation is intentionally simplified for explanation. A real submission must use the benchmark's official scorer and version because seemingly minor normalization choices, multiple-reference handling, Unicode processing, or tokenization can change rankings.

Single-task benchmarks are especially useful when developing task-specific architectures, debugging output constraints, or measuring progress in a well-defined deployment setting. Their main limitation is external validity: high SQuAD performance does not imply reliable open-domain question answering, and high CoNLL-2003 F1 does not imply robust entity recognition in clinical notes, social media, or another language.

#### **Multi-Task Benchmarks** {#multi-task-benchmarks}

A multi-task benchmark combines several tasks so that a model cannot appear broadly capable by specializing in one narrow dataset. This is different from multi-task training. A model may train separately on every benchmark task, fine-tune one shared model across tasks, or use zero-shot prompting; the benchmark's adaptation protocol decides which setting is allowed.

[GLUE](https://aclanthology.org/W18-5446/) brought together diverse natural language understanding tasks and a diagnostic set under a shared platform. [SuperGLUE](https://proceedings.neurips.cc/paper_files/paper/2019/hash/4496bf24afe7fab6f046bf4923da8de6-Abstract.html) followed with a more difficult collection after rapid progress on GLUE. For representation models, [MTEB](https://docs.mteb.org/overview/) evaluates embeddings through task families such as retrieval, classification, clustering, semantic similarity, and pair classification. These suites measure a performance profile, even when a leaderboard compresses that profile into one number.

The benchmark must first produce one score $s_j$ for each task $j$. If tasks use different units or metric directions, scores may be normalized:

$$
\widetilde{s}_j=
\begin{cases}
\dfrac{s_j-l_j}{u_j-l_j}, & \text{if higher is better},\\[6pt]
\dfrac{u_j-s_j}{u_j-l_j}, & \text{if lower is better}.
\end{cases}
$$

$l_j$ and $u_j$ are declared lower and upper reference points for task $j$. The second form reverses metrics such as error rate or latency so that larger normalized values consistently mean better performance. An aggregate can then be computed as:

$$
B=\frac{\sum_{j=1}^{J}w_j\widetilde{s}_j}{\sum_{j=1}^{J}w_j}
$$

$J$ is the number of tasks and $w_j$ is the weight assigned to each task. Equal $w_j$ gives every task equal influence, regardless of test-set size. Weighting by sample count instead gives large datasets more influence. Neither rule is neutral; it encodes what the benchmark considers important.

An overall average can hide radically different models:

| Model | Sentiment | NLI | Retrieval | Macro average |
|---|---:|---:|---:|---:|
| Model A | 90 | 90 | 40 | 73.3 |
| Model B | 75 | 75 | 70 | 73.3 |

Model A is stronger on classification but weak on retrieval. Model B is more balanced. If a product depends on retrieval, equal averages do not make the models interchangeable. A benchmark report should retain per-task scores, worst-task performance, dispersion, and task coverage beside the headline score.

<details>
<summary>Python: Aggregate a multi-task benchmark without hiding coverage or weak tasks</summary>

```python
import pandas as pd


results = pd.DataFrame(
    [
        # Bounds and direction are part of the benchmark specification.
        {"task": "sentiment", "score": 0.90, "lower": 0.50, "upper": 1.00,
         "higher_is_better": True, "weight": 1.0},
        {"task": "nli", "score": 0.84, "lower": 1 / 3, "upper": 1.00,
         "higher_is_better": True, "weight": 1.0},
        {"task": "retrieval_mrr", "score": 0.61, "lower": 0.00, "upper": 1.00,
         "higher_is_better": True, "weight": 1.0},
        {"task": "generation_error", "score": 0.18, "lower": 0.00, "upper": 0.50,
         "higher_is_better": False, "weight": 1.0},
    ]
)

expected_tasks = {"sentiment", "nli", "retrieval_mrr", "generation_error"}

# Step 1: refuse partial or duplicated benchmark submissions.
assert set(results["task"]) == expected_tasks
assert not results["task"].duplicated().any()

# Step 2: map every metric to a declared 0-to-1 direction and scale.
high_better = (
    (results["score"] - results["lower"])
    / (results["upper"] - results["lower"])
)
low_better = (
    (results["upper"] - results["score"])
    / (results["upper"] - results["lower"])
)
results["normalized"] = high_better.where(
    results["higher_is_better"],
    low_better,
).clip(0, 1)

# Step 3: report the headline score together with its hidden structure.
macro_score = results["normalized"].mean()
weighted_score = (
    (results["normalized"] * results["weight"]).sum()
    / results["weight"].sum()
)

report = {
    "macro_score": macro_score,
    "weighted_score": weighted_score,
    "worst_task": results.loc[results["normalized"].idxmin(), "task"],
    "worst_task_score": results["normalized"].min(),
    "task_gap": results["normalized"].max() - results["normalized"].min(),
    "coverage": len(results) / len(expected_tasks),
}

print(results[["task", "score", "normalized"]])
print(report)
```
</details>

Normalization is only defensible when the bounds have a clear meaning. Arbitrary min-max scaling based on whichever models happen to be on a current leaderboard makes scores change when a new model is submitted. Stable reference points, original task scores, and aggregation rules should all be published.

Multi-task benchmarks increase breadth, but they can still be narrow in language, domain, interaction style, and metric choice. A collection of English classification datasets is broader than one classifier test, yet it is not a complete test of multilingual generation, dialogue, safety, factuality, or real-world utility.

#### **Holistic and Capability-Based Benchmarks** {#holistic-and-capability-based-benchmarks}

Traditional benchmark suites often begin by collecting established datasets. Capability-based benchmarking begins one level higher: first decide which capabilities, contexts, users, and risks should be observed, and then select tasks and metrics that cover that design space. The difference is between asking "Which datasets can we run?" and "What evidence would distinguish the behaviors we care about?"

> ![HELM contrasts dataset collections with a taxonomy of scenarios and metrics](assets/benchmark-helm-taxonomy.png)
>
> Previous benchmark collections commonly attach one canonical metric to each dataset. HELM instead decomposes a scenario by task, domain, population, time, and language, then evaluates multiple output properties. The unfilled branches make missing coverage visible rather than silently treating the implemented subset as complete.
>
> Source: [Holistic Evaluation of Language Models, Figure 2](https://arxiv.org/abs/2211.09110)

[HELM](https://crfm.stanford.edu/helm/index.html) is a clear example of this approach. In the original framework, a scenario is not just a dataset name; it represents a use context, and model adaptation is standardized before several metrics are measured. Accuracy remains important, but robustness, calibration, fairness, bias, toxicity, and efficiency reveal tradeoffs that accuracy alone cannot express.

Instead of assigning model $f$ one scalar, a holistic benchmark produces a vector:

$$
\mathbf{s}(f,c)=
\left[
s_{\text{quality}},
s_{\text{robustness}},
s_{\text{calibration}},
s_{\text{fairness}},
s_{\text{safety}},
s_{\text{efficiency}}
\right]
$$

$c$ denotes the evaluated scenario and each component measures a different desired property. Two models may be incomparable: one is more accurate but slower and less robust; another sacrifices a small amount of accuracy for lower latency and more stable subgroup behavior. Averaging every dimension immediately would hide the decision a real user must make.

| Capability question | Suitable scenario variation | Evidence beyond average accuracy |
|---|---|---|
| Does the model understand paraphrases? | Preserve meaning while changing wording or syntax | Consistency and performance drop |
| Can it use knowledge rather than answer format cues? | Counterbalanced options and adversarial distractors | Accuracy by perturbation type |
| Is generation safe across user groups? | Matched prompts that vary demographic references | Toxicity, refusal quality, and subgroup gaps |
| Can confidence support downstream decisions? | Questions across difficulty levels and domains | Calibration error and selective accuracy |
| Is it deployable under a resource budget? | Fixed hardware, batch size, and context length | Latency, throughput, memory, and energy proxies |

Capability-based evaluation is especially useful for general-purpose language models because the same text interface supports many applications. Its cost is complexity: capability labels can be vague, social properties can be contested, automatic judges can introduce another model's bias, and exhaustive coverage is impossible. A responsible holistic benchmark explicitly states which parts of the capability space remain unmeasured.

The practical lesson is not that every study needs hundreds of tests. It is that benchmark coverage should be selected from the intended claim. A paper claiming a better sentiment classifier may need focused in-domain, shifted-domain, robustness, subgroup, and efficiency tests. A paper claiming a generally better language model needs much broader evidence.

#### **Domain-Specific and Multilingual Benchmarks** {#domain-specific-and-multilingual-benchmarks}

General-purpose benchmarks often underrepresent the terminology, document structure, user goals, and error costs of real domains. A model that performs well on movie reviews may fail on clinical negation; a model that extracts organizations from news may misread legal party names; a fluent general model may produce a medically plausible but dangerous answer. Domain-specific benchmarks narrow the population so that the labels and metrics can reflect expert use.

| Benchmark family | Coverage | Why a general benchmark is insufficient |
|---|---|---|
| [BLURB](https://microsoft.github.io/BLURB/) | Biomedical NER, relation extraction, sentence similarity, classification, and QA | Biomedical terminology, evidence conventions, and task distributions differ from open-domain text |
| [LegalBench](https://arxiv.org/abs/2308.11462) | Legal reasoning tasks contributed by legal and NLP researchers | Correct reasoning may depend on legal concepts, jurisdiction, and expert interpretation |
| Financial NLP benchmarks | Reports, filings, numerical QA, sentiment, and risk language | Financial polarity and materiality differ from ordinary sentiment |
| Internal deployment benchmark | Real product traffic, policies, failure severity, and latency | Public datasets rarely match the exact user population and operating constraints |

Expert involvement is needed not only to label examples but also to define the target. In clinical NLP, inter-annotator disagreement may reflect genuinely ambiguous evidence rather than careless annotation. In law, an answer can depend on jurisdiction or assumptions omitted from a short prompt. A benchmark that forces one gold label should document how ambiguity was resolved.

Multilingual benchmarking adds another dimension. The [XTREME benchmark](https://proceedings.mlr.press/v119/hu20b.html), for example, was designed to evaluate cross-lingual generalization across multiple tasks and languages rather than treating English performance as a proxy for multilingual ability. A multilingual protocol must distinguish several settings:

| Setting | Training or prompting data | Test data | Main question |
|---|---|---|---|
| Monolingual evaluation | Target-language data | Same target language | How well does the model serve this language directly? |
| Zero-shot cross-lingual transfer | Usually source-language supervision | Unseen target languages | Does learned knowledge transfer across languages? |
| Translate-train | Translated supervision | Native or translated target data | How useful is machine translation for adaptation? |
| Translate-test | Source-language model | Test inputs translated into the source language | Can translation plus a source-language model solve the task? |
| Multilingual joint training | Data from several languages | Seen and unseen languages | Does joint learning improve broad and low-resource coverage? |

Translation-based test sets support controlled comparisons, but translationese may simplify syntax or erase culture-specific phenomena. Independently authored test sets are more natural but make exact cross-language difficulty matching harder. Neither source is universally superior, so benchmark provenance should identify which construction was used.

If $S_\ell$ is a score for language $\ell$ and $L$ languages are included, a language-macro score is:

$$
S_{\text{macro-lang}}=\frac{1}{L}\sum_{\ell=1}^{L}S_\ell
$$

Every language receives equal weight regardless of test-set size. Two complementary statistics are:

$$
S_{\text{worst-lang}}=\min_{\ell}S_\ell,
\qquad
G_{\text{lang}}=\max_{\ell}S_\ell-\min_{\ell}S_\ell
$$

$S_{\text{worst-lang}}$ exposes the least-served language, while $G_{\text{lang}}$ measures the range between strongest and weakest performance. These values prevent a strong English score or several closely related high-resource languages from hiding severe failures elsewhere.

<details>
<summary>Python: Report multilingual performance without letting large test sets dominate</summary>

```python
import pandas as pd


scores = pd.DataFrame(
    [
        {"language": "English", "family": "Germanic", "n": 5000, "score": 0.91},
        {"language": "German", "family": "Germanic", "n": 1200, "score": 0.86},
        {"language": "Spanish", "family": "Romance", "n": 1400, "score": 0.84},
        {"language": "Swahili", "family": "Atlantic-Congo", "n": 500, "score": 0.58},
        {"language": "Arabic", "family": "Afro-Asiatic", "n": 700, "score": 0.64},
    ]
)

# Step 1: a micro average weights languages by their number of test examples.
micro = (scores["score"] * scores["n"]).sum() / scores["n"].sum()

# Step 2: a language macro average gives each language one vote.
language_macro = scores["score"].mean()

# Step 3: average within families first so one heavily represented family
# does not dominate the summary simply because it contains more languages.
family_scores = scores.groupby("family")["score"].mean()
family_macro = family_scores.mean()

# Step 4: retain distribution-sensitive diagnostics beside every average.
report = {
    "micro_by_examples": micro,
    "macro_by_language": language_macro,
    "macro_by_family": family_macro,
    "worst_language": scores.loc[scores["score"].idxmin(), "language"],
    "worst_language_score": scores["score"].min(),
    "language_gap": scores["score"].max() - scores["score"].min(),
}

print(scores.sort_values("score"))
print(family_scores.sort_values())
print(report)
```
</details>

Domain and multilingual benchmarks improve relevance, but they do not automatically guarantee fairness. Coverage by language name can still omit dialects, code-switching, literacy levels, local entities, and culturally specific tasks. Results should therefore be disaggregated along dimensions supported by the data rather than relying only on a multilingual average.

#### **Benchmark Saturation** {#benchmark-saturation}

A benchmark is saturated when leading systems approach the benchmark's effective ceiling so closely that additional score gains provide little information about meaningful capability differences. The ceiling may be a human estimate, annotation consistency, metric resolution, or the easiest strategy permitted by dataset artifacts. Saturation does not mean the underlying NLP problem has been solved.

> ![Normalized benchmark performance approaches and crosses estimated human performance over time](assets/benchmark-saturation.png)
>
> Several influential benchmarks moved from their initial systems toward estimated human performance within a relatively short period. Crossing the horizontal reference line means surpassing that benchmark's human estimate under its scoring rules, not acquiring human-level language ability.
>
> Source: [Dynabench: Rethinking Benchmarking in NLP, Figure 1](https://arxiv.org/abs/2104.14337)

Let $s_0$ be the initial reference score, $s_h$ an estimated human score, and $s$ the current model score. Normalized progress toward that reference can be written as:

$$
P=\frac{s-s_0}{s_h-s_0}
$$

$P=0$ corresponds to the initial reference, $P=1$ reaches the human estimate, and $P>1$ exceeds it. This normalization is descriptive, not proof of equivalence between human and model behavior. The human estimate may come from annotators with limited instructions, while a model may exploit lexical artifacts or memorize related examples.

Common signs of saturation include:

- score differences smaller than random-seed or confidence-interval variation;
- many submissions tied near the top while subgroup failures remain large;
- only a handful of test errors, so one annotation decision changes rank;
- repeated improvements produced by benchmark-specific prompting or ensembling;
- strong in-distribution scores but large drops on challenge or shifted test sets;
- no relationship between leaderboard gains and downstream utility.

When saturation appears, simply adding harder examples is not always enough. A useful redesign asks why the old test stopped discriminating. The solution may be a fresh hidden set, a temporal split, adversarially collected failures, diagnostic categories, stricter contamination controls, expert review, or metrics that measure robustness and cost as well as correctness.

Static and dynamic benchmarks make different tradeoffs:

| Design | Strength | Main risk | Appropriate use |
|---|---|---|---|
| Fixed public test set | Easy reproduction and historical comparison | Contamination and adaptive overfitting accumulate | Stable research baselines and transparent error analysis |
| Fixed hidden test set | Limits direct test inspection | Repeated submissions still leak information through scores | Competitions and controlled leaderboards |
| Periodically refreshed benchmark | Tracks changing data and models | Scores across versions require careful interpretation | Production monitoring and rapidly changing domains |
| Dynamic human-and-model-in-the-loop benchmark | Collects examples targeting current weaknesses | Higher cost and changing difficulty | Robustness discovery and frontier-model stress testing |

Dynabench proposed human-and-model-in-the-loop collection: annotators create valid examples that fool current systems, those failures become new evaluation or training data, and later systems face updated rounds. Old rounds should still be retained, because solving a new failure mode while forgetting old ones is not progress.

The best response to saturation is usually a benchmark portfolio rather than a single replacement. Keep the stable test for longitudinal comparison, add challenge sets for known weaknesses, include fresh data for current generalization, and report real deployment measurements when the claim concerns practical use.

#### **Data Contamination and Leaderboard Overfitting** {#data-contamination-and-leaderboard-overfitting}

Benchmark validity assumes that test answers provide new evidence about generalization. Data contamination breaks that assumption when evaluation content, labels, or close variants influence model training or selection. In modern NLP, contamination can occur at several levels:

| Leakage path | Example | Why the score becomes misleading |
|---|---|---|
| Within-dataset leakage | Near-duplicate articles appear in train and test | The model recognizes content rather than generalizing to new documents |
| Pretraining contamination | Public benchmark questions and answers occur in web-scale pretraining data | Memorization or benchmark familiarity contributes to apparent zero-shot ability |
| Fine-tuning contamination | Evaluation examples enter instruction-tuning or preference data | The task is no longer unseen even without explicit benchmark labels |
| Prompt-development leakage | Test examples guide prompt wording, demonstrations, or decoding settings | The test set functions as a validation set |
| Leaderboard leakage | Repeated submissions reveal which changes improve hidden-test score | Teams adapt to test-set noise without seeing individual labels |

Exact duplicate detection is necessary but insufficient. Templates, paraphrases, answer explanations, translated versions, and passages containing the answer can transfer benchmark information without identical strings. The NAACL study [Investigating Data Contamination in Modern Benchmarks for Large Language Models](https://aclanthology.org/2024.naacl-long.482/) illustrates both retrieval-based overlap analysis and behavioral tests for cases where model training data is unavailable.

For two texts $a$ and $b$, a simple near-duplicate signal is Jaccard similarity over their $n$-gram sets:

$$
J(a,b)=\frac{|G_n(a)\cap G_n(b)|}{|G_n(a)\cup G_n(b)|}
$$

$G_n(a)$ is the set of contiguous $n$-token sequences in text $a$. The numerator counts shared $n$-grams, and the denominator counts all unique $n$-grams appearing in either text. $J=1$ means the sets are identical and $J=0$ means they share none. Longer $n$-grams identify copied passages more precisely; shorter $n$-grams are more sensitive but produce more accidental matches.

<details>
<summary>Python: Audit train-test near duplicates with normalized word n-grams</summary>

```python
import re
import unicodedata


def normalize_for_overlap(text):
    # Step 1: normalize Unicode, lowercase, and keep word-like tokens.
    text = unicodedata.normalize("NFKC", text).lower()
    return re.findall(r"\w+", text, flags=re.UNICODE)


def word_ngrams(text, n=5):
    tokens = normalize_for_overlap(text)
    return {
        tuple(tokens[start:start + n])
        for start in range(len(tokens) - n + 1)
    }


def jaccard(left, right):
    if not left and not right:
        return 1.0
    if not left or not right:
        return 0.0
    return len(left & right) / len(left | right)


def find_suspicious_pairs(train_texts, test_texts, n=5, threshold=0.50):
    # Step 2: precompute fingerprints so tokenization is performed once.
    train_sets = [word_ngrams(text, n=n) for text in train_texts]
    test_sets = [word_ngrams(text, n=n) for text in test_texts]
    matches = []

    # Step 3: compare every pair for a small audit dataset.
    # At web scale, replace this quadratic loop with MinHash/LSH or retrieval.
    for test_index, test_set in enumerate(test_sets):
        for train_index, train_set in enumerate(train_sets):
            similarity = jaccard(test_set, train_set)
            if similarity >= threshold:
                matches.append(
                    {
                        "train_index": train_index,
                        "test_index": test_index,
                        "jaccard": similarity,
                    }
                )

    # Step 4: review high-overlap pairs rather than deleting them blindly.
    return sorted(matches, key=lambda row: row["jaccard"], reverse=True)


train_examples = [
    "The battery lasts for ten hours after a complete charge.",
    "The support team answered my request within one day.",
]
test_examples = [
    "After a complete charge, the battery lasts for ten hours.",
    "Customer support was friendly but could not solve the problem.",
]

print(find_suspicious_pairs(train_examples, test_examples, n=3, threshold=0.30))
```
</details>

Overlap is evidence for review, not an automatic verdict. Formulaic legal text, common definitions, or short sentences can overlap naturally. Conversely, semantic paraphrases may carry the same answer while sharing few words. A serious audit combines exact hashing, lexical retrieval, semantic retrieval, metadata and timestamp checks, and manual inspection.

Leaderboard overfitting is related but does not require literal data leakage. Suppose submission $k$ has true generalization performance $\mu_k$, while its observed hidden-test score is:

$$
\widehat{s}_k=\mu_k+\varepsilon_k
$$

$\varepsilon_k$ is sampling noise or a benchmark-specific fluctuation. If researchers submit many variants and retain the largest $\widehat{s}_k$, they select for both high $\mu_k$ and unusually positive $\varepsilon_k$. Therefore, $\max_k\widehat{s}_k$ tends to overestimate the selected model's performance on fresh data. This is the same reason a test set loses its independence when it is queried adaptively.

Practical protections include document-level deduplication before splitting, date-based test sets, private or rotating tests, submission limits, public development sets that differ from final tests, fresh confirmation sets, data-provenance records, and reporting how often the benchmark influenced model or prompt selection. The theory of the [reusable holdout](https://pubmed.ncbi.nlm.nih.gov/26250683/) formalizes why unrestricted adaptive reuse damages validity.

No single safeguard solves every form of leakage:

| Risk | Useful detection | Useful mitigation |
|---|---|---|
| Exact duplicate | Cryptographic hashes | Deduplicate before splitting |
| Near duplicate | N-gram, MinHash, or semantic retrieval | Group related documents into one split |
| Web pretraining exposure | Corpus search, timestamps, behavioral probes | Fresh/private tests and transparent contamination audits |
| Prompt overfitting | Prompt history and independent confirmation set | Freeze prompts before final evaluation |
| Leaderboard overfitting | Submission history and public-private score gap | Limit feedback and rotate hidden tests |
| Benchmark-specific shortcut | Challenge sets and counterfactual examples | Redesign data collection and report diagnostic slices |

The benchmark types in this section answer complementary questions:

| Benchmark type | Best question | Main limitation |
|---|---|---|
| Single-task | Can the model solve one clearly specified NLP task? | Narrow external validity |
| Multi-task | Does performance transfer across several task formulations? | Aggregation can hide weak tasks |
| Holistic/capability-based | What profile of capabilities, risks, and costs does the system exhibit? | Expensive and inevitably incomplete |
| Domain-specific | Does the system work under specialist language and error costs? | May not generalize beyond that domain |
| Multilingual | Does behavior transfer across languages and linguistic communities? | Coverage and test comparability remain uneven |
| Dynamic/refreshed | Can evaluation keep discovering weaknesses as models improve? | Longitudinal comparison becomes more complex |

A benchmark score is strongest when the specification is fixed before testing, the baseline ladder is competitive, the data matches the claim, uncertainty and subgroup behavior are visible, and leakage has been audited. Only after those conditions are established does it become meaningful to ask whether a result is state of the art.

### **State of the Art in NLP** {#state-of-the-art-in-nlp}

State of the art, usually abbreviated as **SOTA**, is often used as if it were a permanent title attached to a model. In research, it is better understood as a conditional empirical claim: among the systems compared so far, one system obtained the strongest reported result under a particular benchmark version, metric, adaptation protocol, resource setting, and time.

This distinction follows directly from the previous section. A benchmark defines the measurement environment; a SOTA claim identifies the current leading result inside that environment. If the environment changes, the claim may change as well:

$$
\text{SOTA claim}
=
\text{system}
+
\text{benchmark conditions}
+
\text{comparison evidence}
+
\text{time}
$$

A model can therefore be leaderboard SOTA without being reproducibly better, practically preferable, or resource-efficient. This section separates those meanings so that "SOTA" becomes the beginning of an analysis rather than the end of one.

#### **What Does SOTA Actually Mean?** {#what-does-sota-actually-mean}

A precise SOTA statement should answer: best at what, measured where, under which rules, compared with whom, and as of when? The phrase "our model achieves state-of-the-art NER performance" is too broad. A defensible version is closer to: "under the official CoNLL-2003 English test split and entity-level F1 scorer, our fine-tuned system obtained the highest reported mean among the included comparable systems as of the experiment date."

The coordinates of the claim include:

| Coordinate | Question that must be fixed | Example |
|---|---|---|
| Task and population | Which language behavior and text population are measured? | English newswire named entity recognition |
| Benchmark version | Which data, split, labels, and evaluator are used? | CoNLL-2003 with the official entity-level scorer |
| Metric | Which objective determines "best"? | Macro-F1, Exact Match, BLEU, MRR, or human preference |
| Adaptation | What supervision and tools are allowed? | Full fine-tuning, zero-shot prompting, retrieval, or external APIs |
| Resource constraints | Is compute, latency, memory, cost, or model size bounded? | At most 8 GB memory and 50 ms p95 latency |
| Comparison set | Which baselines and prior systems are included? | Published systems with the same protocol |
| Statistical unit | Is the result a best run, mean over seeds, or confidence interval? | Mean and standard deviation over five seeds |
| Time and access | When was the comparison made and which model version was available? | Checkpoint or API snapshot evaluated on a stated date |

Let $\mathcal{F}(C)$ be the set of systems that satisfy comparison constraints $C$. A constrained SOTA system can be expressed as:

$$
f^*=\underset{f\in\mathcal{F}(C)}{\arg\max}\;
\mathbb{E}_{r\sim\mathcal{R}}\left[\widehat{M}(f;r)\right]
$$

$f$ is a candidate system, $C$ contains rules such as the allowed training data and latency limit, $\widehat{M}$ is the benchmark score, and $r$ represents randomness from initialization, data order, sampling, or prompt selection. The expectation asks for typical performance across the declared source of variation, rather than the luckiest run. If lower values are better, as with perplexity or latency, $\arg\min$ replaces $\arg\max$.

In practice, many papers do not observe the true expectation. They run a small number of experiments and estimate it with the mean:

$$
\overline{s}=\frac{1}{K}\sum_{k=1}^{K}s_k,
\qquad
s_{\mathrm{sd}}=sqrt{\frac{1}{K-1}\sum_{k=1}^{K}(s_k-\overline{s})^2}
$$

$K$ is the number of independent runs, $s_k$ is the score from run $k$, $\overline{s}$ is the sample mean, and $s_{\mathrm{sd}}$ describes run-to-run variability. A candidate with $90.3\pm0.5$ is not clearly superior to a baseline with $90.1\pm0.2$ merely because 90.3 is larger. The distributions overlap, and the observed difference may be smaller than ordinary training variation.

Several valid forms of SOTA can coexist:

| SOTA form | Optimization target | Example claim |
|---|---|---|
| Absolute SOTA | Highest primary benchmark score without a resource restriction | Best entity-level F1 on the official test set |
| Constrained SOTA | Highest score among systems satisfying a fixed budget | Best F1 below 100 ms latency |
| Data-efficient SOTA | Best score at a fixed number of labeled examples | Best 32-shot intent classification accuracy |
| Domain SOTA | Best result for a specified text population | Best clinical NER score on one hospital dataset |
| Pareto SOTA | Not dominated across performance and one or more costs | No other model is both more accurate and cheaper |

These claims answer different research questions. A 0.2-point score improvement obtained with five times the inference cost may be absolute SOTA, while a slightly weaker distilled model may be constrained or Pareto SOTA. Neither description is dishonest if its conditions are explicit.

The deepest limitation is that SOTA is benchmark-relative. [The Benchmark Lottery](https://arxiv.org/abs/2107.07002) demonstrates that changing the subset of benchmark tasks can alter algorithm rankings. A method aligned with the community's selected tasks may look generally superior even when another selection would favor a different method. "Best on this benchmark" should never silently become "best at language."

#### **Leaderboard SOTA** {#leaderboard-sota}

Leaderboard SOTA is the highest accepted score on an evaluation server or public result table. Leaderboards are useful infrastructure: they standardize the scorer, protect hidden labels, preserve historical comparisons, and make progress easy to inspect. They are also an incomplete scientific instrument because rank compresses every experimental choice into one ordered number.

A leaderboard entry may differ from another entry in model size, external data, number of ensemble members, retrieval access, prompt tuning, test-time computation, or hyperparameter-search budget. Unless tracks separate these conditions, the first row answers "which submitted system achieved the highest score?" rather than "which algorithm is intrinsically better?"

Let the observed difference between a candidate and the previous leader be:

$$
\widehat{\Delta}=\widehat{s}_{\text{candidate}}-\widehat{s}_{\text{leader}}
$$

$\widehat{\Delta}>0$ is sufficient to order two displayed scores when higher is better. It is not sufficient to establish that the candidate has a reliably higher population performance. Both scores contain test-sample uncertainty, training randomness, implementation variation, and possible adaptation to the leaderboard.

Hyperparameter search creates another asymmetry. If one model receives many more trials, its best observed run has more opportunities to benefit from favorable configurations and random noise.

> ![Expected validation accuracy changes with hyperparameter search budget](assets/sota-hyperparameter-budget.png)
>
> In this sentiment-classification example, logistic regression has better expected validation performance under a small search budget, while the CNN becomes better when more hyperparameter assignments are available. Reporting only each model's best run hides the amount of search required to obtain it.
>
> Source: [Show Your Work: Improved Reporting of Experimental Results, Figure 1](https://aclanthology.org/D19-1224/)

If $H$ hyperparameter configurations produce validation scores $v_1,\ldots,v_H$, model selection reports:

$$
v_{\max}=\max_{1\le h\le H}v_h
$$

As $H$ increases, $v_{\max}$ tends to increase even when the underlying search space has not improved. The increase can reflect discovery of a genuinely better configuration, favorable noise, or both. Fair leaderboard comparisons should therefore report tuning strategy and budget, not just final hyperparameters.

A paired bootstrap can test whether the candidate's test predictions consistently outperform the leader on resampled versions of the same examples. Pairing is important because both systems face the same inputs.

<details>
<summary>Python: Estimate uncertainty around a claimed leaderboard improvement</summary>

```python
import numpy as np
from sklearn.metrics import f1_score


def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average="macro", zero_division=0)


def paired_bootstrap_difference(
    y_true,
    leader_predictions,
    candidate_predictions,
    metric_fn=macro_f1,
    rounds=5000,
    seed=42,
):
    y_true = np.asarray(y_true)
    leader_predictions = np.asarray(leader_predictions)
    candidate_predictions = np.asarray(candidate_predictions)

    assert len(y_true) == len(leader_predictions) == len(candidate_predictions)
    rng = np.random.default_rng(seed)

    # Step 1: calculate the observed candidate-minus-leader difference.
    observed = (
        metric_fn(y_true, candidate_predictions)
        - metric_fn(y_true, leader_predictions)
    )

    # Step 2: resample shared test positions and preserve paired predictions.
    differences = []
    for _ in range(rounds):
        indices = rng.integers(0, len(y_true), size=len(y_true))
        differences.append(
            metric_fn(y_true[indices], candidate_predictions[indices])
            - metric_fn(y_true[indices], leader_predictions[indices])
        )

    differences = np.asarray(differences)
    lower, upper = np.percentile(differences, [2.5, 97.5])

    # Step 3: report an interval instead of reducing the result to rank alone.
    return {
        "observed_difference": float(observed),
        "bootstrap_95pct_interval": (float(lower), float(upper)),
        "fraction_of_resamples_candidate_better": float(
            np.mean(differences > 0)
        ),
    }


# Replace these short arrays with predictions from the locked test run.
y_true = ["pos", "neg", "neutral", "pos", "neg", "neutral", "pos", "neg"]
leader = ["pos", "neg", "neutral", "neutral", "neg", "pos", "pos", "neg"]
candidate = ["pos", "neg", "neutral", "pos", "neutral", "neutral", "pos", "neg"]

print(paired_bootstrap_difference(y_true, leader, candidate))
```
</details>

This interval measures sensitivity to the sampled test examples under the chosen metric. It does not include variability across random seeds, prompts, checkpoints, human raters, or model API versions. Those sources require repeated runs or hierarchical analysis rather than pretending one bootstrap solves every uncertainty.

Rank can also change for reasons unrelated to model quality:

| Cause | What changes | Better reporting practice |
|---|---|---|
| Metric rounding | Very small differences appear tied or ordered | Publish unrounded scores and uncertainty |
| Random seed | Fine-tuning converges to different solutions | Report all seeds, mean, and dispersion |
| Prompt selection | Instruction wording changes model behavior | Freeze and publish templates and demonstrations |
| Search budget | One method receives more trials | Match or disclose the budget |
| Ensemble or self-consistency | Test-time compute increases | Report members, samples, tokens, and cost |
| External data or retrieval | Additional knowledge enters the system | Use separate tracks or explicit flags |
| API update | The same model name points to changed behavior | Record snapshot/version and evaluation date |

Leaderboard SOTA is valuable for discovering candidates and tracking a shared challenge. A strong paper then moves beyond rank: it reproduces comparable baselines, quantifies uncertainty, explains the source of improvement, and tests whether the conclusion survives other data slices and constraints.

#### **Reproducible SOTA** {#reproducible-sota}

Reproducible SOTA means that the superiority claim can be reconstructed and remains credible when the experiment is rerun under the documented conditions. Releasing a final score is not enough. Another researcher needs the artifacts that generated the inputs, model state, predictions, and score.

Related terms are often used differently across fields, but the following separation is practical for NLP:

| Level | Main question | Typical requirement |
|---|---|---|
| Repeatability | Can the original team obtain the result again? | Same code, data, environment, and protocol |
| Computational reproducibility | Can another team rerun the released pipeline? | Complete artifacts, versions, and instructions |
| Independent reproduction | Does an independent implementation support the conclusion? | Recreated method and comparable evaluation |
| Replication/generalization | Does the finding persist in new settings? | New datasets, languages, domains, or populations |

A result can be repeatable but not reproducible if it depends on an unavailable checkpoint or private preprocessing. It can be computationally reproducible but fail to generalize if the same code reproduces the score only on one artifact-heavy dataset. These are different strengths of evidence.

Neural NLP pipelines contain many hidden state variables:

- dataset revision, filtering, split IDs, and label mapping;
- tokenizer files, vocabulary, normalization, and maximum sequence length;
- pretrained checkpoint name and exact revision;
- initialization, data-loader, sampling, and generation seeds;
- optimizer, scheduler, batch construction, gradient accumulation, and stopping rule;
- package, CUDA, driver, and hardware versions;
- prompt templates, demonstrations, ordering, system messages, and decoding parameters;
- retrieval corpus snapshot, chunking, index version, and reranker;
- evaluator implementation, metric version, human instructions, or judge-model snapshot;
- raw predictions, failed requests, retry policy, and post-processing.

The paper [Show Your Work](https://aclanthology.org/D19-1224/) emphasizes that test scores alone cannot reveal the computational process used to find a model. Reproduction studies have likewise observed result variation under different seeds, environments, and dependency versions. Reproducible SOTA therefore reports both the final artifact and the search process that produced it.

For repeated runs, the candidate should be compared against a strong baseline under matched conditions. Reporting the five candidate seeds but copying the baseline's best score from another paper creates an asymmetric comparison. Ideally, both systems use the same data, evaluator, number of seeds, stopping policy, and tuning budget.

An experiment manifest can make the main dependencies machine-readable. Hashes identify exact files; they do not prove data quality, but they reveal accidental substitutions.

<details>
<summary>Python: Build a reproducibility manifest for an NLP evaluation run</summary>

```python
from datetime import datetime, timezone
import hashlib
from importlib.metadata import PackageNotFoundError, version
import json
from pathlib import Path
import platform


def sha256_file(path):
    """Fingerprint an artifact without embedding its contents in the report."""
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def installed_version(package_name):
    try:
        return version(package_name)
    except PackageNotFoundError:
        return None


# Step 1: record scientific choices, not only software versions.
manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "task": "three-class sentiment classification",
    "benchmark": {
        "name": "ProductSentiment",
        "version": "1.0",
        "test_split_sha256": sha256_file("data/test.jsonl"),
        "primary_metric": "macro_f1",
        "evaluator_commit": "7d3f1a2",
    },
    "model": {
        "checkpoint": "distilbert-base-uncased",
        "checkpoint_revision": "<immutable-model-commit>",
        "code_commit": "<git-commit-used-for-this-run>",
    },
    "training": {
        "seeds": [13, 21, 34, 55, 89],
        "epochs": 5,
        "learning_rate": 2e-5,
        "batch_size": 32,
        "selection_rule": "highest development macro_f1",
        "hyperparameter_trials": 12,
    },
    "tokenization": {
        "max_length": 256,
        "truncation": True,
        "padding": "longest-in-batch",
    },
    "environment": {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "torch": installed_version("torch"),
        "transformers": installed_version("transformers"),
        "scikit_learn": installed_version("scikit-learn"),
    },
    "outputs": {
        "predictions_sha256": sha256_file("results/test_predictions.jsonl"),
        "full_seed_scores": "results/seed_scores.json",
    },
}

# Step 2: store the manifest beside the immutable predictions and configuration.
Path("results/run_manifest.json").write_text(
    json.dumps(manifest, indent=2, sort_keys=True),
    encoding="utf-8",
)
```
</details>

For API-based language models, an exact reproduction may be impossible if the provider changes the model or infrastructure. The correct response is not to omit details, but to record everything observable: provider, endpoint, model snapshot, request date, system prompt, parameters, retries, raw responses, and evaluator version. The result can then be interpreted as a time-stamped measurement rather than a timeless property.

A reproducible SOTA claim is stronger when:

| Evidence | Why it matters |
|---|---|
| All seed-level scores | Exposes instability and selective reporting |
| Raw test predictions | Allows independent metric and error analysis |
| Immutable model and data revisions | Prevents silent artifact drift |
| Full configuration and environment | Makes hidden defaults visible |
| Search space and trial budget | Separates algorithm quality from tuning effort |
| Comparable baseline reruns | Avoids comparing a new optimized pipeline with stale literature numbers |
| Independent reproduction | Tests whether success depends on the original environment or team knowledge |
| Cross-dataset confirmation | Tests whether the conclusion survives beyond one benchmark |

Reproducibility does not require every researcher to afford the largest experiment. It requires the claim to reveal its requirements honestly. If reproducing a result needs hundreds of accelerators, proprietary data, or a closed model, that limitation is part of the scientific result.

#### **Practical SOTA** {#practical-sota}

Practical SOTA is the system that best satisfies the objective and constraints of a real use case. It may not have the highest public benchmark score. Production text arrives from different users, domains, time periods, devices, and languages; systems also face latency, cost, privacy, safety, maintenance, and reliability requirements that a benchmark may not measure.

Consider an emotion classifier for a mental-health support platform. A leaderboard might reward the highest average macro-F1 on a curated English dataset. Deployment may instead require high recall for distress-related categories, reliable behavior on code-switched and informal text, calibrated uncertainty, on-premise inference, p95 latency below 100 ms, and a human escalation path. A model that gains 0.4 macro-F1 but violates privacy or misses the latency target is not practically superior.

The cleanest formulation is constrained optimization:

$$
\begin{aligned}
\max_f \quad & Q(f) \\
\text{subject to}\quad
& L_{p95}(f)\le L_{\max},\\
& C_{1k}(f)\le C_{\max},\\
& \operatorname{Mem}(f)\le M_{\max},\\
& R_g(f)\ge R_{\min}\quad \text{for every critical group }g.
\end{aligned}
$$

$Q(f)$ is the chosen quality measure, $L_{p95}$ is 95th-percentile latency, $C_{1k}$ is cost per 1,000 inputs, $\operatorname{Mem}(f)$ is peak memory, and $R_g$ is a required reliability measure such as recall for critical group $g$. The constraints eliminate unusable models before quality is compared. This is often clearer than combining everything into one utility score with arbitrary weights.

Practical evaluation should mirror the complete system. A retrieval-augmented QA service includes query rewriting, retrieval, reranking, prompt construction, generation, citation extraction, safety filters, and retries. Reporting only the generator's score ignores the pipeline that users experience. Similarly, a classifier's end-to-end latency includes tokenization, batching, data transfer, and post-processing rather than only a single GPU forward pass.

<details>
<summary>Python: Select the strongest deployable NLP model under explicit constraints</summary>

```python
import pandas as pd


models = pd.DataFrame(
    [
        {"model": "large-transformer", "macro_f1": 0.918,
         "worst_domain_f1": 0.79, "p95_latency_ms": 180,
         "peak_memory_gb": 11.2, "cost_per_1k": 0.82,
         "on_premise": True},
        {"model": "distilled-transformer", "macro_f1": 0.906,
         "worst_domain_f1": 0.84, "p95_latency_ms": 54,
         "peak_memory_gb": 2.8, "cost_per_1k": 0.19,
         "on_premise": True},
        {"model": "hosted-api", "macro_f1": 0.924,
         "worst_domain_f1": 0.81, "p95_latency_ms": 310,
         "peak_memory_gb": 0.0, "cost_per_1k": 1.40,
         "on_premise": False},
        {"model": "tfidf-linear", "macro_f1": 0.874,
         "worst_domain_f1": 0.82, "p95_latency_ms": 8,
         "peak_memory_gb": 0.4, "cost_per_1k": 0.02,
         "on_premise": True},
    ]
)

constraints = {
    "max_p95_latency_ms": 100,
    "max_peak_memory_gb": 4.0,
    "max_cost_per_1k": 0.25,
    "require_on_premise": True,
    "min_worst_domain_f1": 0.80,
}

# Step 1: remove systems that cannot satisfy the deployment contract.
feasible = models[
    (models["p95_latency_ms"] <= constraints["max_p95_latency_ms"])
    & (models["peak_memory_gb"] <= constraints["max_peak_memory_gb"])
    & (models["cost_per_1k"] <= constraints["max_cost_per_1k"])
    & (models["worst_domain_f1"] >= constraints["min_worst_domain_f1"])
]

if constraints["require_on_premise"]:
    feasible = feasible[feasible["on_premise"]]

# Step 2: optimize robustness first, then average quality, among usable systems.
ranking = feasible.sort_values(
    ["worst_domain_f1", "macro_f1"],
    ascending=False,
)

if ranking.empty:
    raise RuntimeError("No candidate satisfies the deployment constraints")

selected = ranking.iloc[0]
print("Practical SOTA:", selected["model"])
print(ranking)
```
</details>

In this hypothetical case, the hosted API has the highest benchmark score but violates latency, cost, privacy, and deployment requirements. The distilled model becomes practical SOTA because it is the strongest feasible system and has better worst-domain behavior than the alternatives.

Practical SOTA also changes over the system lifecycle. Input language drifts, new product categories appear, abuse strategies adapt, API prices change, and hardware is replaced. A production leader should therefore be evaluated continuously with fresh traffic samples, delayed labels, drift indicators, critical error rates, and rollback criteria.

| Research-only comparison | Practical comparison |
|---|---|
| One fixed test distribution | In-domain, shifted, temporal, and stress-test distributions |
| Mean benchmark quality | Critical-class recall, worst-group quality, and calibration |
| Model-only runtime | End-to-end p50/p95/p99 latency and throughput |
| Parameter count as efficiency proxy | Measured memory, hardware-specific latency, energy, and cost |
| One checkpoint | Monitoring, update policy, rollback, and incident behavior |
| Public data availability | Privacy, license, governance, and auditability |

A practical claim should not replace benchmark reporting. Public benchmarks make research comparable; deployment tests make decisions relevant. The strongest evidence reports both and explains where their conclusions agree or diverge.

#### **Efficiency and Pareto-Optimal Models** {#efficiency-and-pareto-optimal-models}

Efficiency-aware evaluation treats performance and resource use as separate objectives. A larger model may gain a fraction of a benchmark point while requiring much more memory, training compute, inference latency, or energy. Choosing the largest score alone assumes every additional unit of performance is worth any cost, which is rarely true in research or deployment.

Costs also belong to different lifecycle phases:

| Phase | Representative costs | Why it matters |
|---|---|---|
| Development | Pretraining, fine-tuning, failed runs, hyperparameter search, data processing | Determines research accessibility and the total cost of finding the model |
| Deployment | Latency, throughput, accelerator memory, energy and price per request | Accumulates with every processed document or generated token |
| Maintenance | Monitoring, retraining, indexing, migration, and evaluation | Determines whether the system remains useful as data and infrastructure change |

Parameter count and FLOPs are useful hardware-independent proxies, but they are not substitutes for measured latency or energy. Two models with similar FLOPs can use hardware differently because of sequence length, memory access, sparsity, batching, quantization, and implementation quality. A fair report states hardware, precision, batch size, input length, generation length, warm-up, and whether preprocessing is included.

> ![A performance-cost Pareto frontier separates dominated systems from efficient trade-offs](assets/sota-pareto-front.png)
>
> Models on the frontier represent different efficient trade-offs between performance and FLOPs. A point below and to the right of another point is dominated: another model is both stronger and cheaper. A Pareto improvement can be scientifically valuable even when it does not create a new absolute highest score.
>
> Source: [On the Concept of Resource-Efficiency in NLP, Figure 1](https://aclanthology.org/2023.nodalida-1.15/)

For quality $Q$ where higher is better and a cost vector $\mathbf{c}$ where lower is better, model $a$ dominates model $b$ when:

$$
Q(a)\ge Q(b),
\qquad
c_j(a)\le c_j(b)\;\;\text{for every cost }j,
$$

with at least one strict inequality. In plain language, $a$ is no worse on any objective and better on at least one. A model is Pareto-optimal when no evaluated alternative dominates it. The set of all such models forms the Pareto frontier.

This definition avoids arbitrary exchange rates. A weighted utility such as accuracy minus $0.01\times$ latency can rank models, but the coefficient encodes one user's preferences. The Pareto frontier first removes clearly inferior choices; users can then select among frontier models according to their actual constraints.

<details>
<summary>Python: Compute and plot a multi-cost Pareto frontier for NLP models</summary>

```python
import matplotlib.pyplot as plt
import pandas as pd


models = pd.DataFrame(
    [
        {"model": "linear", "macro_f1": 0.86, "latency_ms": 4, "memory_gb": 0.3},
        {"model": "small", "macro_f1": 0.90, "latency_ms": 18, "memory_gb": 1.4},
        {"model": "distilled", "macro_f1": 0.915, "latency_ms": 42, "memory_gb": 2.8},
        {"model": "medium", "macro_f1": 0.919, "latency_ms": 95, "memory_gb": 6.2},
        {"model": "large", "macro_f1": 0.923, "latency_ms": 210, "memory_gb": 13.0},
        {"model": "inefficient-run", "macro_f1": 0.907, "latency_ms": 120, "memory_gb": 8.0},
    ]
)


def dominates(left, right):
    """Higher F1 is better; lower latency and memory are better."""
    no_worse = (
        left["macro_f1"] >= right["macro_f1"]
        and left["latency_ms"] <= right["latency_ms"]
        and left["memory_gb"] <= right["memory_gb"]
    )
    strictly_better = (
        left["macro_f1"] > right["macro_f1"]
        or left["latency_ms"] < right["latency_ms"]
        or left["memory_gb"] < right["memory_gb"]
    )
    return no_worse and strictly_better


# Step 1: mark a model as dominated if any competitor is no worse everywhere.
dominated_by = []
for index, candidate in models.iterrows():
    dominators = [
        competitor["model"]
        for other_index, competitor in models.iterrows()
        if other_index != index and dominates(competitor, candidate)
    ]
    dominated_by.append(dominators)

models["dominated_by"] = dominated_by
models["pareto_optimal"] = models["dominated_by"].map(len).eq(0)
frontier = models[models["pareto_optimal"]].sort_values("latency_ms")

# Step 2: visualize one cost dimension while Pareto status uses both costs.
for _, row in models.iterrows():
    color = "tab:blue" if row["pareto_optimal"] else "lightgray"
    plt.scatter(row["latency_ms"], row["macro_f1"], color=color)
    plt.annotate(row["model"], (row["latency_ms"], row["macro_f1"]))

plt.plot(frontier["latency_ms"], frontier["macro_f1"], color="tab:blue")
plt.xlabel("p95 latency (ms, lower is better)")
plt.ylabel("Macro-F1 (higher is better)")
plt.title("Performance-latency view of the Pareto frontier")
plt.show()

print(models[["model", "pareto_optimal", "dominated_by"]])
```
</details>

The paper [Green AI](https://arxiv.org/abs/1907.10597) argues that computational price should be reported beside performance. A simple development-cost approximation is:

$$
C_{\text{dev}}\propto E_T\,D_T\,H
$$

$E_T$ is the cost of one training execution unit, $D_T$ represents how much training data or how many update units are processed, and $H$ represents the number of model-development trials, including hyperparameter search. Two papers can train the same final architecture once, yet incur very different development costs because one searched hundreds of configurations.

For a deployed system processing $N$ inputs, a simplified lifetime cost is:

$$
C_{\text{life}}(N)=C_{\text{dev}}+N\,C_{\text{inf}}
$$

$C_{\text{inf}}$ is average end-to-end inference cost per input. A compressed model may require an expensive distillation stage but save cost on every request. If model $A$ has higher development cost but lower inference cost than model $B$, its break-even volume is:

$$
N^*=\frac{C_{\text{dev},A}-C_{\text{dev},B}}
{C_{\text{inf},B}-C_{\text{inf},A}}
$$

After approximately $N^*$ inputs, the accumulated inference savings offset the additional development cost. The formula assumes stable costs and comparable quality; real analyses should also include maintenance, hardware utilization, and expected system lifetime.

Efficiency reporting should match the claim:

| Claim | Minimum useful measurements |
|---|---|
| Faster training | Time and compute to reach a fixed quality, including search |
| Smaller model | Parameters, checkpoint size, peak memory, and quality |
| Faster inference | End-to-end latency distribution and throughput on stated hardware |
| Cheaper generation | Input/output tokens, requests, retries, and monetary cost |
| Lower energy use | Measured energy, hardware utilization, location, and workload |
| Data-efficient learning | Learning curve across matched labeled-data budgets |

Pareto SOTA broadens what counts as progress. A method that matches existing quality with half the latency is meaningful even if it does not increase the top score. Conversely, an absolute SOTA point far beyond the existing cost range should be described honestly as a new performance extreme, not automatically as the best model for every user.

#### **How to Read a SOTA Claim Critically** {#how-to-read-a-sota-claim-critically}

A SOTA claim should be read as a structured argument. The paper proposes that a system is better; the benchmark, baselines, statistical analysis, resource report, and artifacts are evidence. Critical reading asks whether that evidence supports the exact wording of the claim.

Start by rewriting the headline in conditional form:

> Under benchmark version $B$, metric $M$, adaptation protocol $A$, comparison set $P$, resource setting $C$, and evaluation date $t$, system $f$ obtained result $s$.

Anything missing from that sentence is a possible source of ambiguity. The following questions form a practical audit:

| Question | Warning sign | Stronger evidence |
|---|---|---|
| What is the claim's scope? | "Best NLP model" from one English dataset | Task-, domain-, language-, and version-specific wording |
| Are baselines competitive? | Only weak or outdated references | Rerun strong classical, neural, and pretrained baselines |
| Are protocols matched? | Different data, prompts, retrieval, or tuning budgets | Same splits, evaluator, adaptation rules, and budgets |
| Is the gain larger than uncertainty? | One best seed or rounded 0.1-point gain | Seed distributions, paired intervals, and effect size |
| Was the test set protected? | Repeated prompt or model selection on test feedback | Frozen choices and an independent final set |
| Does the metric measure the desired behavior? | One overlap metric for open-ended generation | Complementary automatic, human, and diagnostic evaluation |
| Is improvement broad or localized? | Only an aggregate score is shown | Per-task, subgroup, difficulty, and error-category results |
| Is the result reproducible? | Missing code, checkpoint, prompt, or evaluator version | Immutable artifacts, raw predictions, and manifests |
| What did the improvement cost? | No search, hardware, latency, or token report | Development and deployment resource measurements |
| Does it transfer beyond the benchmark? | No shifted-domain or temporal evaluation | Cross-dataset, robustness, and deployment-aligned tests |

Suppose a paper reports 92.4 macro-F1 against a previous 92.1. The new system uses the best of 30 seeds, while the baseline is copied from a paper that used one seed. Its paired 95% interval for the test difference is $[-0.1,0.6]$, p95 latency is three times higher, and no predictions are released. The correct conclusion is not "the result is useless," but that different levels of evidence support different statements:

- It may be a **reported leaderboard improvement** because 92.4 is the highest displayed score.
- It is not yet a **reliable performance improvement** because the interval includes zero and selection budgets are asymmetric.
- It is not yet **reproducible SOTA** because the artifacts and run distribution are incomplete.
- It is not **practical SOTA** for latency-sensitive use without showing that the quality gain justifies the cost.

Evidence strength can be viewed as a ladder:

| Evidence level | What can reasonably be claimed |
|---|---|
| Highest single reported run | Candidate leaderboard result |
| Mean and variation under matched seeds | More stable within-pipeline advantage |
| Paired uncertainty and controlled tuning budget | Evidence that the benchmark gain is not ordinary noise |
| Released artifacts and independent reproduction | Reproducible benchmark advantage |
| Cross-domain, multilingual, or temporal confirmation | Broader generalization evidence |
| Deployment-aligned quality, safety, and resource tests | Practical superiority for the specified use case |

Error analysis is essential even when the aggregate gain is statistically reliable. A candidate may improve frequent easy examples while worsening rare critical labels. For affective computing, a higher average emotion score can coexist with lower recall for distress, sarcasm, or code-switched text. The direction of improvement matters as much as its average size.

SOTA should therefore be reported as a profile rather than a trophy:

| Dimension | Core result |
|---|---|
| Leaderboard | Official score and rank under the exact benchmark version |
| Reliability | Per-run scores, uncertainty, and matched comparison budget |
| Reproducibility | Code, data/checkpoint revisions, configuration, and predictions |
| Generalization | Domain, language, subgroup, robustness, and temporal results |
| Practicality | End-to-end quality, latency, cost, safety, privacy, and maintenance |
| Efficiency | Development cost, inference cost, and Pareto status |

The phrase "state of the art" is most informative when it identifies a boundary of current evidence: the best result observed under explicit conditions. It becomes misleading when that local boundary is presented as universal superiority. The next step is therefore to design comparisons in which data, evaluation, tuning, randomness, and reporting are fair enough for any SOTA conclusion to be defensible.

### **Designing a Fair Comparison** {#designing-a-fair-comparison}

A fair comparison is an experiment in which the intended difference between systems is allowed to vary while alternative explanations are controlled, matched, or reported. If a new encoder receives more labeled data, a larger search budget, a different test split, and a more favorable evaluator than its baseline, the final score cannot isolate the value of the encoder itself.

The logic is close to a controlled experiment:

| Experimental concept | NLP comparison |
|---|---|
| Treatment | The method change being tested, such as a new loss or architecture |
| Control | A strong baseline without that change |
| Outcome | A predeclared benchmark metric and diagnostic measures |
| Experimental unit | Document, conversation, user, query, or another independent unit |
| Controlled conditions | Data, preprocessing, evaluator, tuning policy, seeds, and hardware rules |
| Nuisance variables | Random initialization, implementation details, API drift, and annotator variation |

Let $P$ denote the shared experimental protocol and let $m_{\text{new}}$ and $m_{\text{base}}$ differ only in the component under investigation. The target comparison is:

$$
\Delta(P)=
\mathbb{E}\!\left[M(m_{\text{new}};P)\right]
-
\mathbb{E}\!\left[M(m_{\text{base}};P)\right]
$$

$M$ is the evaluation measure and the expectations cover declared randomness. Interpreting $\Delta(P)$ as the effect of the new component is plausible only when the rest of $P$ is shared. The conclusion also remains protocol-specific: changing the data population or resource constraint can change $\Delta$.

Fairness does not mean forcing every system into identical internals. A linear classifier and a Transformer naturally have different optimizers. It means giving each method a defensible opportunity under a clearly defined comparison regime, then exposing any remaining asymmetry.

#### **Same Data and Data Splits** {#same-data-and-data-splits}

All compared systems should learn from and be evaluated on the same examples unless data access is explicitly the research variable. "We use the same dataset" is not precise enough. Dataset versions can differ in corrected labels, removed duplicates, licenses, split assignments, and preprocessing. Even one changed test example can affect a close leaderboard result.

The split must protect the correct unit of independence. NLP datasets often contain correlated observations:

- several sentences from one document;
- multiple turns from one conversation;
- reviews written by the same user or describing the same product;
- paraphrases generated from one source sentence;
- articles copied across websites;
- repeated templates with different entity values;
- records from the same patient, author, speaker, or time period.

If correlated examples cross the train-test boundary, the model may identify the source, style, or template instead of generalizing to a genuinely new unit. Sentence-level random splitting is therefore inappropriate when the intended deployment unit is a new document or user.

| Generalization question | Appropriate split | NLP example |
|---|---|---|
| New examples from the same population | Random or stratified split | Balanced intent classification from one stable collection |
| New documents, users, or conversations | Group-aware split | Hold out complete authors or dialogue threads |
| Future language use | Temporal split | Train on reviews before 2025 and test on later reviews |
| New domains or genres | Domain-held-out split | Train on product reviews and test on restaurant reviews |
| New languages | Language-held-out split | Train multilingual representations without target-language labels |
| Model selection with limited data | Nested or group-aware cross-validation | Tune only inside training folds and reserve outer folds for evaluation |

The following official scikit-learn visualizations show why the split algorithm must match the data-generating process:

| Group-aware evaluation | Time-aware evaluation |
|:---:|:---:|
| ![GroupKFold keeps each group in one side of a fold](assets/fair-comparison-group-kfold.png) | ![TimeSeriesSplit trains only on earlier observations](assets/fair-comparison-time-series-split.png) |
| Complete groups move between training and testing folds. | Each training window precedes its corresponding test window. |

Source: [Visualizing cross-validation behavior in scikit-learn](https://scikit-learn.org/stable/auto_examples/model_selection/plot_cv_indices.html)

For `GroupKFold`, the colors in the final row identify groups, and every group is assigned wholly to training or testing within a fold. `TimeSeriesSplit` grows the training window from earlier samples and evaluates on later samples. Shuffling before a temporal split would allow future language patterns to inform a model evaluated on the past.

A three-way split has distinct responsibilities:

| Split | Allowed use | Prohibited use |
|---|---|---|
| Training | Fit model parameters and preprocessing learned from data | Estimate final generalization performance |
| Development/validation | Select hyperparameters, prompts, checkpoints, and thresholds | Repeatedly support final claims as if unseen |
| Test | One final comparison after choices are frozen | Model, prompt, threshold, or error-driven feature selection |

Preprocessing must be fitted inside the training boundary. TF-IDF vocabulary, normalization statistics, label-frequency weights, feature selection, and learned tokenizers can all leak test information if computed before splitting. Labels are not the only source of leakage; observing test text can reveal vocabulary, topic, length, or domain information.

<details>
<summary>Python: Create and audit a group-aware NLP split</summary>

```python
import hashlib
import json

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold


# Each row is an utterance, but conversation_id is the independent unit.
examples = pd.DataFrame(
    {
        "example_id": [f"ex-{index:03d}" for index in range(20)],
        "conversation_id": [f"conv-{index // 2:02d}" for index in range(20)],
        "text": [f"Example utterance {index}" for index in range(20)],
        "label": ["positive", "negative", "neutral", "positive"] * 5,
    }
)

# Step 1: reserve complete conversations for the final test set.
outer_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)
train_dev_index, test_index = next(
    outer_split.split(
        examples,
        y=examples["label"],
        groups=examples["conversation_id"],
    )
)

train_dev = examples.iloc[train_dev_index].reset_index(drop=True)
test = examples.iloc[test_index].reset_index(drop=True)

# Step 2: tune with stratified group folds inside train+development only.
inner_cv = StratifiedGroupKFold(
    n_splits=4,
    shuffle=True,
    random_state=42,
)
inner_folds = list(
    inner_cv.split(
        train_dev,
        y=train_dev["label"],
        groups=train_dev["conversation_id"],
    )
)

# Step 3: fail immediately if any conversation crosses the outer boundary.
train_groups = set(train_dev["conversation_id"])
test_groups = set(test["conversation_id"])
assert train_groups.isdisjoint(test_groups)
assert set(train_dev["example_id"]).isdisjoint(test["example_id"])

# Step 4: fingerprint ordered test IDs so later experiments use the same split.
test_id_payload = "\n".join(test["example_id"]).encode("utf-8")
split_record = {
    "split_method": "GroupShuffleSplit",
    "group_column": "conversation_id",
    "random_state": 42,
    "test_ids": test["example_id"].tolist(),
    "test_ids_sha256": hashlib.sha256(test_id_payload).hexdigest(),
}

print(json.dumps(split_record, indent=2))
```
</details>

The tiny dataset is only for illustrating the workflow. Real splits should inspect class coverage and group sizes, and very rare classes may require manual or optimization-based assignment. The correct response to a difficult split is not to let entities leak, but to report the limitation and collect enough independent units.

Before training, record dataset revision, collection dates, example IDs, group field, split code, random state, and split hashes. Every model in the comparison should consume those locked artifacts.

#### **Same Evaluation Protocol** {#same-evaluation-protocol}

Two systems can receive the same test examples and still be evaluated differently. The evaluation protocol includes every transformation between raw input and final score:

$$
x
\xrightarrow{\;P\;}
x'
\xrightarrow{\;A,f\;}
z
\xrightarrow{\;G\;}
\widehat{y}
\xrightarrow{\;M(\widehat{y},y)\;}
s
$$

$P$ is preprocessing and input construction, $A$ is the allowed adaptation procedure, $f$ is the model, $z$ is raw model output, $G$ is decoding and post-processing, $\widehat{y}$ is the final prediction, $M$ is the evaluator, and $s$ is the score. A fair comparison either shares these components or identifies which component is part of the method being tested.

Metric names alone do not define a protocol:

| Reported label | Choices that can change the value |
|---|---|
| F1 | Binary, micro, macro, weighted, token-level, or entity-level averaging |
| BLEU | Tokenization, case handling, smoothing, reference set, and implementation signature |
| ROUGE | Stemming, sentence splitting, variant, reference aggregation, and package version |
| Exact Match | Unicode, punctuation, article, whitespace, and multiple-reference normalization |
| Perplexity | Tokenizer, stride, context window, token count, and treatment of special tokens |
| LLM-as-judge score | Judge model/version, rubric, order, temperature, and retry policy |

For sequence labeling, one system must not receive token-level F1 while another receives strict entity-level F1. For generation, one model must not use a longer maximum output, more beam candidates, or external retrieval unless those resources are permitted for every system or declared as the tested intervention.

Prompts are part of the protocol. A zero-shot comparison should use the same task instruction, label descriptions, output schema, demonstration policy, temperature, maximum tokens, and parser. When one model requires a different chat template, report the exact template and explain how semantic equivalence was maintained.

Post-processing deserves equal scrutiny. Mapping unknown labels to the majority class, discarding failed generations, retrying malformed answers, or extracting only text inside a pattern can improve a score. The evaluator should count failures according to a predeclared rule rather than silently removing them.

<details>
<summary>Python: Centralize the evaluator and reject incompatible submissions</summary>

```python
from dataclasses import dataclass
import hashlib
import json

from sklearn.metrics import classification_report, f1_score


@dataclass(frozen=True)
class EvaluationProtocol:
    benchmark_version: str
    labels: tuple[str, ...]
    primary_metric: str
    average: str
    failed_output_policy: str
    evaluator_version: str

    def fingerprint(self):
        payload = json.dumps(self.__dict__, sort_keys=True).encode("utf-8")
        return hashlib.sha256(payload).hexdigest()


PROTOCOL = EvaluationProtocol(
    benchmark_version="ProductSentiment-1.0",
    labels=("negative", "neutral", "positive"),
    primary_metric="macro_f1",
    average="macro",
    failed_output_policy="count_as_incorrect",
    evaluator_version="sentiment-eval-1.2.0",
)


def evaluate_submission(gold_rows, prediction_rows, protocol):
    # Step 1: align by immutable ID rather than relying on row order.
    gold = {row["example_id"]: row["label"] for row in gold_rows}
    predictions = {row["example_id"]: row.get("prediction") for row in prediction_rows}
    assert set(gold) == set(predictions), "Missing or extra prediction IDs"

    y_true = []
    y_pred = []
    failures = 0

    # Step 2: apply one declared failure policy to every system.
    for example_id in sorted(gold):
        target = gold[example_id]
        prediction = predictions[example_id]

        if prediction not in protocol.labels:
            failures += 1
            # A reserved invalid label is always counted as incorrect.
            prediction = "__INVALID_OUTPUT__"

        y_true.append(target)
        y_pred.append(prediction)

    # Step 3: compute every score through the same implementation.
    score = f1_score(
        y_true,
        y_pred,
        labels=list(protocol.labels),
        average=protocol.average,
        zero_division=0,
    )

    return {
        "protocol_sha256": protocol.fingerprint(),
        "macro_f1": score,
        "failed_outputs": failures,
        "per_class": classification_report(
            y_true,
            y_pred,
            labels=list(protocol.labels),
            output_dict=True,
            zero_division=0,
        ),
    }
```
</details>

One central evaluator should score saved predictions from every system. This design prevents each model script from implementing its own slightly different metric and makes later audits possible. For established benchmarks, use the official evaluator directly and record its version or commit.

#### **Hyperparameter and Compute Budget** {#hyperparameter-and-compute-budget}

Model quality depends on both the method and the resources spent finding a good configuration. Hyperparameters include learning rate, batch size, regularization, prompt wording, number and order of demonstrations, retrieval depth, decoding temperature, and stopping criteria. Selecting among many alternatives is part of model development, even when no gradient is computed.

The total search cost can be written as:

$$
B_{\text{search}}(m)=\sum_{h=1}^{H_m}C(m,h)
$$

$H_m$ is the number of configurations tried for method $m$, and $C(m,h)$ is the cost of training and evaluating configuration $h$. Cost may be measured in accelerator-hours, wall-clock time, FLOPs, energy, tokens, API price, or another resource relevant to the study.

Equal trial counts and equal compute are different comparison regimes. Ten trials of a large model can cost more than one hundred trials of a linear model. Conversely, allowing only one trial may unfairly penalize a method known to require tuning. The research question should determine the budget rule:

| Budget regime | Rule | Question answered |
|---|---|---|
| Equal configurations | Same number of sampled settings | Which method works better with the same tuning opportunities? |
| Equal compute | Same total accelerator-hours, FLOPs, or monetary cost | Which method uses a fixed development resource most effectively? |
| Equal wall time | Same elapsed development deadline | Which approach can be delivered fastest on available infrastructure? |
| Method-appropriate tuning | Each method receives a justified search space | What performance can each established method reasonably reach? |
| Performance target | Measure cost required to reach a fixed score | Which method reaches useful quality most efficiently? |

No regime is universally fair. A paper should declare the regime, justify it, and report the realized budget. The search space matters as much as the search algorithm: a baseline with a poorly chosen learning-rate range is not strong merely because many trials were run.

All tuning must use training and development data. If cross-validation is used, hyperparameter selection belongs inside the inner folds and performance estimation belongs in the outer folds. Tuning on the final test set turns it into development data.

The expected best development score is a function of budget:

$$
g_m(b)=\mathbb{E}\left[\max_{h:\,B_h\le b}v_{m,h}\right]
$$

$v_{m,h}$ is validation performance for configuration $h$, $B_h$ is cumulative search cost, and $g_m(b)$ describes how well method $m$ is expected to perform after spending budget $b$. Comparing complete budget-performance curves is more informative than comparing two final maxima.

<details>
<summary>Python: Compare hyperparameter searches at an equal compute budget</summary>

```python
import pandas as pd


# Every attempted configuration is logged, including weak and failed trials.
trials = pd.DataFrame(
    [
        {"model": "tfidf_logreg", "trial": 1, "gpu_hours": 0.02, "dev_f1": 0.701},
        {"model": "tfidf_logreg", "trial": 2, "gpu_hours": 0.02, "dev_f1": 0.714},
        {"model": "tfidf_logreg", "trial": 3, "gpu_hours": 0.02, "dev_f1": 0.709},
        {"model": "transformer", "trial": 1, "gpu_hours": 0.80, "dev_f1": 0.792},
        {"model": "transformer", "trial": 2, "gpu_hours": 0.75, "dev_f1": 0.817},
        {"model": "transformer", "trial": 3, "gpu_hours": 0.85, "dev_f1": 0.808},
        {"model": "domain_adapted", "trial": 1, "gpu_hours": 1.10, "dev_f1": 0.821},
        {"model": "domain_adapted", "trial": 2, "gpu_hours": 1.05, "dev_f1": 0.833},
        {"model": "domain_adapted", "trial": 3, "gpu_hours": 1.15, "dev_f1": 0.829},
    ]
).sort_values(["model", "trial"])

# Step 1: reconstruct the score available after each amount of spending.
trials["cumulative_gpu_hours"] = trials.groupby("model")["gpu_hours"].cumsum()
trials["best_dev_f1_so_far"] = trials.groupby("model")["dev_f1"].cummax()


def best_within_budget(group, budget):
    eligible = group[group["cumulative_gpu_hours"] <= budget]
    if eligible.empty:
        return None
    best_row = eligible.loc[eligible["dev_f1"].idxmax()]
    return {
        "selected_trial": int(best_row["trial"]),
        "best_dev_f1": float(best_row["dev_f1"]),
        "spent_gpu_hours": float(eligible["gpu_hours"].sum()),
        "trials_completed": len(eligible),
    }


# Step 2: apply the same resource ceiling to every model family.
budget = 2.5
comparison = {
    model: best_within_budget(group, budget)
    for model, group in trials.groupby("model")
}

# Step 3: report when a method cannot complete even one trial under the budget.
print(comparison)
print(trials)
```
</details>

The values are hypothetical, but the logging principle is general. Failed runs, early-stopped runs, preprocessing, prompt search, and evaluation calls consume development resources and should not disappear from the budget report.

#### **Multiple Random Seeds** {#multiple-random-seeds}

Modern NLP systems are stochastic. Random initialization, mini-batch order, dropout masks, data subsampling, train-dev splitting, prompt demonstrations, decoding, and distributed kernels can all change results. A single seed confounds method quality with one random trajectory.

Different sources of randomness should be named rather than hidden behind one integer:

| Random source | What it changes | Recommended treatment |
|---|---|---|
| Parameter initialization | Starting point for task-specific weights | Repeat training seeds |
| Data order and augmentation | Optimization path and examples seen together | Control and log sampler seeds |
| Split construction | Which examples define train/dev/test | Lock the test split; analyze split variation separately |
| Few-shot demonstrations | Context examples and order | Repeat or predeclare demonstration sets |
| Stochastic decoding | Generated output | Fix decoding for deterministic tests or repeat generations |
| Human evaluation assignment | Which rater sees which output and order | Randomize with a recorded design and model rater effects |

The study [Measuring the Instability of Fine-Tuning](https://aclanthology.org/2023.acl-long.342/) emphasizes that standard deviation alone captures only one aspect of instability. Two model families can have similar mean scores while disagreeing on which individual examples are correct.

For $K$ seed runs, the sample mean and standard deviation were defined earlier. An estimated standard error of the mean is:

$$
SE(\overline{s})=\frac{s_{\mathrm{sd}}}{\sqrt{K}}
$$

Under an approximate normality assumption, a small-sample confidence interval can use the Student $t$ distribution:

$$
\overline{s}\pm t_{K-1,\,0.975}\frac{s_{\mathrm{sd}}}{\sqrt{K}}
$$

$t_{K-1,0.975}$ is the 97.5th percentile with $K-1$ degrees of freedom. The interval represents uncertainty in the estimated mean across the declared run distribution. With only three or five seeds it can be wide and sensitive to outliers, which is precisely why all seed scores should be shown.

Prediction-level consistency for example $i$ can be defined as:

$$
C_i=\max_{c\in\mathcal{Y}}
\frac{1}{K}\sum_{k=1}^{K}
\mathbb{1}\!\left[\widehat{y}^{(k)}_i=c\right]
$$

$\mathcal{Y}$ is the label set and $\widehat{y}^{(k)}_i$ is the prediction from seed $k$. $C_i=1$ means every run agrees; $C_i=0.6$ means the most common prediction appears in three of five runs. Low-consistency examples identify unstable decision boundaries even when aggregate scores look stable.

<details>
<summary>Python: Summarize seed-level performance and prediction consistency</summary>

```python
from collections import Counter
import math

import pandas as pd
from scipy.stats import t


seed_scores = pd.DataFrame(
    [
        {"model": "baseline", "seed": 13, "macro_f1": 0.812},
        {"model": "baseline", "seed": 21, "macro_f1": 0.819},
        {"model": "baseline", "seed": 34, "macro_f1": 0.807},
        {"model": "baseline", "seed": 55, "macro_f1": 0.816},
        {"model": "baseline", "seed": 89, "macro_f1": 0.811},
        {"model": "candidate", "seed": 13, "macro_f1": 0.826},
        {"model": "candidate", "seed": 21, "macro_f1": 0.831},
        {"model": "candidate", "seed": 34, "macro_f1": 0.814},
        {"model": "candidate", "seed": 55, "macro_f1": 0.829},
        {"model": "candidate", "seed": 89, "macro_f1": 0.824},
    ]
)


def summarize_runs(values, confidence=0.95):
    count = len(values)
    mean = values.mean()
    standard_deviation = values.std(ddof=1)
    standard_error = standard_deviation / math.sqrt(count)
    critical = t.ppf((1 + confidence) / 2, df=count - 1)

    return pd.Series(
        {
            "runs": count,
            "mean": mean,
            "standard_deviation": standard_deviation,
            "ci_lower": mean - critical * standard_error,
            "ci_upper": mean + critical * standard_error,
            "minimum": values.min(),
            "maximum": values.max(),
        }
    )


# Step 1: retain the full run table and derive a summary from it.
summary = (
    seed_scores.groupby("model")["macro_f1"]
    .apply(summarize_runs)
    .unstack()
)

# Rows are seed runs and columns are the same ordered test examples.
candidate_predictions = [
    ["pos", "neg", "neutral", "pos"],
    ["pos", "neg", "neutral", "neutral"],
    ["pos", "neutral", "neutral", "pos"],
    ["pos", "neg", "neutral", "pos"],
    ["pos", "neg", "positive", "pos"],
]

# Step 2: measure how often the modal label is reproduced for each example.
consistency = []
for predictions_for_example in zip(*candidate_predictions):
    most_common_count = Counter(predictions_for_example).most_common(1)[0][1]
    consistency.append(most_common_count / len(candidate_predictions))

print(seed_scores)
print(summary)
print("Per-example consistency:", consistency)
print("Mean prediction consistency:", sum(consistency) / len(consistency))
```
</details>

Matched seed lists help standardize the number of opportunities, but seed 13 in two architectures does not guarantee equivalent optimization noise. The primary report should include each model's run distribution, and paired reasoning should be justified rather than assumed.

Best-seed reporting is acceptable only when explicitly labeled and accompanied by the selection rule and all attempted runs. For scientific comparison, mean, dispersion, and individual values should remain visible.

#### **Statistical Significance** {#statistical-significance}

Statistical significance asks whether the observed difference would be surprising under a specified null hypothesis. It does not measure the size, practical importance, or probability that a method is "truly better." A tiny effect can become statistically significant with a large test set, while an important effect can remain uncertain with too few independent units.

For systems $A$ and $B$, a common null hypothesis is:

$$
H_0:\;\Delta=M(A)-M(B)=0
$$

The alternative may be two-sided, $H_1:\Delta\ne0$, or directional, $H_1:\Delta>0$, if the direction was justified before observing results. Choosing a one-sided test after seeing a favorable direction inflates evidence.

The **evaluation unit** is central. Predictions are paired because both systems process the same examples. However, utterances from one conversation or sentences from one document are not independent. Resampling or permutation should occur at the document or conversation level when that is the independent unit.

> ![NLPStatTest separates system building, data analysis, hypothesis testing, effect size, power, and reporting](assets/fair-comparison-statistical-testing-workflow.png)
>
> A sound comparison does not jump directly from two scores to a p-value. It defines evaluation units, checks assumptions, chooses an appropriate paired procedure, estimates effect size and uncertainty, considers power, and reports the full design.
>
> Source: [NLPStatTest: A Toolkit for Comparing NLP System Performance, Figure 1](https://aclanthology.org/2020.aacl-demo.7/)

The appropriate test depends on the output structure, metric, and assumptions. [The Hitchhiker's Guide to Testing Statistical Significance in NLP](https://aclanthology.org/P18-1128/) provides a detailed NLP-specific decision process.

| Comparison setting | Possible procedure | Important condition |
|---|---|---|
| Paired correctness for classification | McNemar's exact or asymptotic test | Each independent unit gives paired binary outcomes |
| Non-decomposable corpus metric | Paired bootstrap or approximate randomization | Resample or swap the correct independent units |
| Per-unit numeric scores | Paired $t$-test, permutation, or Wilcoxon signed-rank | Check distribution and symmetry assumptions |
| Multiple seeds | Model run-level differences or hierarchical analysis | Do not treat test examples and seeds as the same randomness |
| Human ratings | Mixed-effects models or clustered bootstrap | Account for both item and rater dependence |
| Many tasks or model comparisons | Family-wise or false-discovery correction | Declare the family of hypotheses |

Approximate randomization tests whether the observed pairing of predictions with system names matters. For each randomization, the two predictions are swapped within randomly selected evaluation units, and the metric difference is recomputed.

If $d_{\mathrm{obs}}$ is the observed difference and $d_b$ is the difference after randomization $b$, a two-sided Monte Carlo p-value is:

$$
p=\frac{1+\sum_{b=1}^{B}\mathbb{1}\!\left[|d_b|\ge|d_{\mathrm{obs}}|\right]}{B+1}
$$

$B$ is the number of randomizations. Adding 1 to numerator and denominator avoids a reported p-value of exactly zero from a finite simulation. A small $p$ indicates that differences at least this extreme were rare under the exchangeability implied by $H_0$.

<details>
<summary>Python: Group-aware approximate randomization for an NLP classifier</summary>

```python
import numpy as np
from sklearn.metrics import f1_score


def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average="macro", zero_division=0)


def approximate_randomization(
    y_true,
    predictions_a,
    predictions_b,
    unit_ids,
    metric_fn=macro_f1,
    rounds=10000,
    seed=42,
):
    y_true = np.asarray(y_true)
    predictions_a = np.asarray(predictions_a)
    predictions_b = np.asarray(predictions_b)
    unit_ids = np.asarray(unit_ids)

    assert len(y_true) == len(predictions_a) == len(predictions_b) == len(unit_ids)
    unique_units = np.unique(unit_ids)
    rng = np.random.default_rng(seed)

    # Step 1: calculate the observed paired metric difference.
    observed = metric_fn(y_true, predictions_b) - metric_fn(y_true, predictions_a)
    extreme = 0

    # Step 2: swap complete documents/conversations, not correlated rows.
    for _ in range(rounds):
        swap_unit = dict(
            zip(unique_units, rng.integers(0, 2, size=len(unique_units)).astype(bool))
        )
        swap_mask = np.array([swap_unit[unit_id] for unit_id in unit_ids])

        randomized_a = predictions_a.copy()
        randomized_b = predictions_b.copy()
        randomized_a[swap_mask] = predictions_b[swap_mask]
        randomized_b[swap_mask] = predictions_a[swap_mask]

        randomized_difference = (
            metric_fn(y_true, randomized_b)
            - metric_fn(y_true, randomized_a)
        )
        extreme += abs(randomized_difference) >= abs(observed)

    # Step 3: return both effect size and evidence against the null.
    return {
        "observed_macro_f1_difference": float(observed),
        "two_sided_p_value": (extreme + 1) / (rounds + 1),
        "randomization_rounds": rounds,
        "independent_units": len(unique_units),
    }


y_true = ["pos", "neg", "neutral", "pos", "neg", "neutral", "pos", "neg"]
model_a = ["pos", "neutral", "neutral", "neg", "neg", "neutral", "pos", "neg"]
model_b = ["pos", "neg", "neutral", "pos", "neutral", "neutral", "pos", "neg"]
conversation_ids = ["c1", "c1", "c2", "c2", "c3", "c3", "c4", "c4"]

print(
    approximate_randomization(
        y_true,
        model_a,
        model_b,
        conversation_ids,
        rounds=5000,
    )
)
```
</details>

A p-value should be accompanied by the observed difference and a confidence interval. The effect answers "how much?"; the interval answers "with what precision?"; the test answers a narrower question about compatibility with $H_0$.

Multiple comparisons require correction. If a study tries many models, tasks, prompts, seeds, and metrics, the chance of at least one favorable result increases. Declare the primary hypothesis before testing. For secondary families, use procedures such as Holm correction for family-wise error or Benjamini-Hochberg for false discovery rate, and report how the family was defined.

Statistical significance is not practical significance. A reliable increase of 0.05 macro-F1 points may not justify a tenfold inference cost, while a small average gain concentrated in a critical safety class may be important. Minimum meaningful effects should be connected to the research or deployment decision.

#### **Reporting Performance, Cost, and Limitations** {#reporting-performance-cost-and-limitations}

A comparison is complete only when readers can reconstruct both its result and its boundary. The main table should not contain only the strongest score. It should expose central tendency, uncertainty, resource use, and relevant diagnostics.

An effective result table might contain:

| System | Primary metric | Run variability | Worst group | Params | Search cost | p95 latency | Peak memory |
|---|---:|---:|---:|---:|---:|---:|---:|
| Majority baseline | Macro-F1 | Deterministic | Group F1 | 0 | Negligible | Measured | Measured |
| TF-IDF + logistic regression | Mean over folds/seeds | SD or CI | Group F1 | Feature dependent | CPU-hours | Measured | Measured |
| Fine-tuned encoder | Mean over seeds | SD and CI | Group F1 | Trainable/total | GPU-hours | Measured | Measured |
| Proposed method | Mean over seeds | SD and CI | Group F1 | Trainable/total | GPU-hours | Measured | Measured |

The table should specify units and hardware in its caption or surrounding text. "Latency = 12" is meaningless without milliseconds, batch size, hardware, precision, input length, and measurement boundary.

Performance reporting should include:

- every primary seed score and the aggregation rule;
- unrounded primary and diagnostic metrics;
- confidence intervals or another uncertainty estimate;
- per-class, per-task, subgroup, and shifted-domain results where relevant;
- failed-output and abstention rates;
- the exact baseline source and whether it was rerun;
- statistical test, evaluation unit, alternative, and correction procedure;
- raw predictions when licensing and privacy permit.

Cost reporting should include:

- number of training and tuning trials, including failed trials;
- hardware type/count, precision, wall time, and accelerator-hours;
- training examples, tokens, steps, and context length;
- parameters updated and total parameters used;
- end-to-end latency distribution, throughput, peak memory, and batch size;
- API tokens, retries, retrieval calls, and monetary cost for hosted systems;
- development cost separately from cost per deployed input.

Limitations are not a ceremonial paragraph. They define where the claim may fail:

| Limitation category | Questions to answer |
|---|---|
| Data | Which domains, periods, dialects, users, and labels are missing? |
| Annotation | How was disagreement handled and which constructs are subjective? |
| Metric | Which quality dimensions are not measured or may be gamed? |
| Contamination | Could training or prompt development have exposed evaluation content? |
| Generalization | Which languages, shifts, and tasks were not tested? |
| Resources | Who can reproduce the training and what costs were omitted? |
| Access | Are checkpoints, APIs, data, or evaluators private or mutable? |
| Harm | Which errors could affect people and which groups may be underserved? |

The [Model Cards for Model Reporting](https://dl.acm.org/doi/10.1145/3287560.3287596) framework is useful for documenting intended use, evaluation conditions, ethical considerations, and subgroup performance. A model card does not replace a paper's experimental report, but it keeps deployment-facing context attached to the artifact.

Negative and inconclusive results also matter. If the proposed method does not consistently beat a strong baseline, report the distribution and analyze why. Hiding an unsuccessful task or seed makes the remaining average look stronger and prevents later researchers from learning where the method is brittle.

The standard for a fair comparison can be summarized as follows:

| Layer | Fixed or matched evidence |
|---|---|
| Data | Version, provenance, independent unit, splits, and preprocessing |
| Protocol | Adaptation, prompt, decoding, post-processing, and evaluator |
| Development | Search space, selection rule, compute budget, and failed trials |
| Randomness | Named sources, seed lists, full run distribution, and stability |
| Statistics | Paired units, effect size, interval, test assumptions, and corrections |
| Reporting | Quality profile, costs, artifacts, limitations, and raw outputs |

### **Case Study: From Baseline to SOTA** {#case-study-from-baseline-to-sota}

The following hypothetical case study shows how the pieces fit together. The task is three-class sentiment classification for product reviews: `negative`, `neutral`, and `positive`. The research question is whether **domain-adaptive pretraining (DAPT)** on unlabeled product reviews improves a pretrained encoder's sentiment performance, particularly on product categories absent from supervised training.

The intended claim is deliberately narrow:

> Under a locked product-group split, equal fine-tuning/search budgets, five training seeds, and macro-F1 as the primary metric, does continued masked-language pretraining on unlabeled in-domain text improve a pretrained sentiment classifier over the same encoder without DAPT?

This wording identifies the intervention, baseline, population, protocol, and outcome. It does not claim a universally better sentiment model.

The benchmark contract is:

| Component | Case-study choice |
|---|---|
| Labeled data | Product reviews with three sentiment labels |
| Independent group | Product ID, preventing one product from crossing splits |
| Outer test | Held-out products and the newest collection period |
| Primary metric | Macro-F1, because class frequencies are imbalanced |
| Diagnostics | Per-class F1, worst-category F1, calibration, and failed outputs |
| Baseline protocol | Same tokenizer, classification head, fine-tuning budget, and seeds |
| Candidate change | Additional masked-language pretraining on training-period product text |
| Resource report | DAPT cost, fine-tuning cost, p95 latency, memory, and parameters |

#### **Establishing the Baseline Ladder** {#establishing-the-baseline-ladder}

The baseline ladder starts with tests that expose dataset structure and advances only when each rung answers a new question:

| Rung | System | Diagnostic question |
|---:|---|---|
| 1 | Majority label | Can any model beat the class prior? |
| 2 | Stratified random | Is performance above a frequency-matched random policy? |
| 3 | Sentiment lexicon with negation | How much can explicit polarity knowledge explain? |
| 4 | Word/character TF-IDF + logistic regression | Are lexical patterns sufficient for strong performance? |
| 5 | Frozen pretrained encoder + linear classifier | Do generic contextual representations help without encoder adaptation? |
| 6 | Fully fine-tuned pretrained encoder | What is the strongest standard pretrained baseline? |
| 7 | DAPT + identical fine-tuning | Does unlabeled in-domain adaptation add value? |

The most important comparison is rung 7 against rung 6. Comparing DAPT only with TF-IDF would show that a large pretrained pipeline is better than a classical baseline, but would not isolate the effect of domain-adaptive pretraining.

Every rung uses the same outer split and official evaluator. Deterministic systems run once; stochastic neural systems run the same five declared seeds. Search spaces are appropriate to each family and their realized costs are reported.

<details>
<summary>Python: Define the baseline ladder and immutable experiment matrix</summary>

```python
from itertools import product
import hashlib
import json


BASELINE_LADDER = [
    {"name": "majority", "family": "data-independent", "stochastic": False},
    {"name": "stratified_random", "family": "data-independent", "stochastic": True},
    {"name": "lexicon_negation", "family": "heuristic", "stochastic": False},
    {"name": "tfidf_logreg", "family": "classical", "stochastic": False},
    {"name": "frozen_encoder", "family": "pretrained-feature", "stochastic": True},
    {"name": "fine_tuned_encoder", "family": "pretrained", "stochastic": True},
    {"name": "dapt_fine_tuned_encoder", "family": "proposed", "stochastic": True},
]

SEEDS = [13, 21, 34, 55, 89]
TEST_SPLIT_SHA256 = "<fingerprint-from-locked-test-ids>"
EVALUATOR_SHA256 = "<fingerprint-from-evaluation-protocol>"

# Step 1: deterministic systems get one run; stochastic systems get all seeds.
jobs = []
for specification in BASELINE_LADDER:
    run_seeds = SEEDS if specification["stochastic"] else [None]
    for seed in run_seeds:
        jobs.append(
            {
                "model": specification["name"],
                "family": specification["family"],
                "seed": seed,
                "test_split_sha256": TEST_SPLIT_SHA256,
                "evaluator_sha256": EVALUATOR_SHA256,
                "status": "planned",
            }
        )

# Step 2: fingerprint the plan before final test evaluation begins.
serialized = json.dumps(jobs, sort_keys=True).encode("utf-8")
experiment_plan = {
    "jobs": jobs,
    "plan_sha256": hashlib.sha256(serialized).hexdigest(),
    "primary_metric": "macro_f1",
    "candidate": "dapt_fine_tuned_encoder",
    "primary_baseline": "fine_tuned_encoder",
}

print(json.dumps(experiment_plan, indent=2))
```
</details>

The immutable plan prevents an unfavorable seed from being quietly dropped after results are seen. Additional runs can still be conducted, but they should be labeled exploratory and applied symmetrically when used for the primary comparison.

#### **Running the Benchmark** {#running-the-benchmark}

Development proceeds without touching final test labels:

1. Fit and tune each baseline using training data and group-aware development folds.
2. Select hyperparameters through the declared budget rule.
3. Freeze preprocessing, checkpoints, thresholds, prompts, and evaluator version.
4. Run every planned seed on the locked test inputs.
5. Save one prediction file and one manifest per run.
6. Score all systems centrally, then compute uncertainty and diagnostic slices.
7. Measure inference cost under a fixed hardware, batch, and input-length protocol.

The DAPT stage may use unlabeled product text only from the training period. Allowing test-period text would introduce transductive information and change the claim. If transductive adaptation is a desired setting, it should be a separate benchmark track available to every comparable method.

<details>
<summary>Python: Framework-style orchestration for the locked benchmark run</summary>

```python
from pathlib import Path
import json
import time


def execute_job(job, train_fn, predict_fn, evaluate_fn, output_root="results"):
    run_name = f"{job['model']}__seed-{job['seed']}"
    run_directory = Path(output_root) / run_name
    run_directory.mkdir(parents=True, exist_ok=False)

    # Step 1: train only from the locked training/development artifacts.
    started = time.perf_counter()
    model, training_record = train_fn(
        model_name=job["model"],
        seed=job["seed"],
        train_path="data/locked/train.jsonl",
        development_path="data/locked/dev.jsonl",
    )
    training_seconds = time.perf_counter() - started

    # Step 2: produce raw predictions without reading final test labels.
    prediction_path = run_directory / "test_predictions.jsonl"
    inference_record = predict_fn(
        model=model,
        input_path="data/locked/test_inputs.jsonl",
        output_path=prediction_path,
    )

    # Step 3: use the same central evaluator for every saved prediction file.
    evaluation = evaluate_fn(
        gold_path="data/private/test_gold.jsonl",
        prediction_path=prediction_path,
        expected_protocol_sha256=job["evaluator_sha256"],
    )

    # Step 4: persist score, cost, and provenance together.
    manifest = {
        **job,
        "training_seconds": training_seconds,
        "training": training_record,
        "inference": inference_record,
        "evaluation": evaluation,
    }
    (run_directory / "manifest.json").write_text(
        json.dumps(manifest, indent=2, sort_keys=True),
        encoding="utf-8",
    )
    return manifest


# `train_fn`, `predict_fn`, and `evaluate_fn` are injected interfaces.
# The same orchestration code is used for every rung of the baseline ladder.
```
</details>

The orchestration function separates training, prediction, and evaluation. In a real repository, deterministic baselines may bypass gradient training, but they should still produce the same prediction and manifest schema. This makes central scoring and cost comparison straightforward.

Quality assurance runs before the headline table is created:

- verify every planned job completed or explain failures;
- verify test IDs and protocol hashes match;
- check that no model has missing or duplicate predictions;
- recompute metrics from raw outputs;
- inspect label distributions and failed-output counts;
- compare run logs with the declared search budget;
- audit a sample of model disagreements and annotation errors;
- measure latency after warm-up with identical batch and length settings.

#### **Comparing Performance and Cost** {#comparing-performance-and-cost}

Assume the locked experiment produces the following hypothetical summary:

| System | Test macro-F1 | Worst-category F1 | p95 latency | Development cost |
|---|---:|---:|---:|---:|
| Majority | 0.244 | 0.000 | 0.1 ms | Negligible |
| Stratified random | 0.329 | 0.301 | 0.2 ms | Negligible |
| Lexicon + negation | 0.548 | 0.421 | 0.8 ms | 1 CPU-hour |
| TF-IDF + logistic regression | 0.731 | 0.662 | 4 ms | 3 CPU-hours |
| Frozen encoder | $0.768\pm0.006$ | 0.701 | 31 ms | 4 GPU-hours |
| Fine-tuned encoder | $0.817\pm0.005$ | 0.754 | 48 ms | 18 GPU-hours |
| DAPT + fine-tuning | $0.829\pm0.006$ | 0.781 | 50 ms | 54 GPU-hours |

These are illustrative values, not results from a real dataset. Their purpose is to show how the conclusion is constructed.

The candidate improves mean macro-F1 over the strongest standard baseline by:

$$
\Delta=0.829-0.817=0.012
$$

The absolute gain is 1.2 percentage points. A relative error reduction can also be reported, using error $e=1-F_1$:

$$
\operatorname{RER}
=
\frac{e_{\text{baseline}}-e_{\text{candidate}}}{e_{\text{baseline}}}
=
\frac{(1-0.817)-(1-0.829)}{1-0.817}
\approx 0.0656
$$

The candidate reduces the remaining macro-F1 error by about 6.6% under this definition. Relative error reduction can look much larger than absolute gain, so both should be reported and the error definition must be explicit.

The worst-category improvement from 0.754 to 0.781 supports the domain-shift motivation better than the average alone. However, DAPT increases development cost from 18 to 54 GPU-hours while inference latency remains nearly unchanged. The candidate is plausible for repeated deployment because the extra cost occurs during adaptation, but whether it is worthwhile depends on model lifetime and the value of the quality gain.

<details>
<summary>Python: Build a quality-cost summary for the case study</summary>

```python
import pandas as pd


runs = pd.DataFrame(
    [
        {"model": "fine_tuned_encoder", "seed": 13, "macro_f1": 0.812},
        {"model": "fine_tuned_encoder", "seed": 21, "macro_f1": 0.819},
        {"model": "fine_tuned_encoder", "seed": 34, "macro_f1": 0.811},
        {"model": "fine_tuned_encoder", "seed": 55, "macro_f1": 0.824},
        {"model": "fine_tuned_encoder", "seed": 89, "macro_f1": 0.819},
        {"model": "dapt_fine_tuned_encoder", "seed": 13, "macro_f1": 0.823},
        {"model": "dapt_fine_tuned_encoder", "seed": 21, "macro_f1": 0.831},
        {"model": "dapt_fine_tuned_encoder", "seed": 34, "macro_f1": 0.821},
        {"model": "dapt_fine_tuned_encoder", "seed": 55, "macro_f1": 0.837},
        {"model": "dapt_fine_tuned_encoder", "seed": 89, "macro_f1": 0.833},
    ]
)

costs = pd.DataFrame(
    [
        {"model": "fine_tuned_encoder", "development_gpu_hours": 18,
         "p95_latency_ms": 48, "worst_category_f1": 0.754},
        {"model": "dapt_fine_tuned_encoder", "development_gpu_hours": 54,
         "p95_latency_ms": 50, "worst_category_f1": 0.781},
    ]
)

# Step 1: keep full runs, then calculate transparent summary statistics.
quality = (
    runs.groupby("model")["macro_f1"]
    .agg(mean="mean", standard_deviation="std", minimum="min", maximum="max")
    .reset_index()
)
summary = quality.merge(costs, on="model", validate="one_to_one")

# Step 2: quantify the primary effect against the declared baseline.
baseline_mean = summary.loc[
    summary["model"] == "fine_tuned_encoder", "mean"
].item()
candidate_mean = summary.loc[
    summary["model"] == "dapt_fine_tuned_encoder", "mean"
].item()

absolute_gain = candidate_mean - baseline_mean
relative_error_reduction = absolute_gain / (1 - baseline_mean)

# Step 3: expose quality and cost together rather than ranking by score only.
comparison = {
    "absolute_macro_f1_gain": absolute_gain,
    "relative_error_reduction": relative_error_reduction,
    "additional_development_gpu_hours": 54 - 18,
    "additional_p95_latency_ms": 50 - 48,
    "worst_category_f1_gain": 0.781 - 0.754,
}

print(summary)
print(comparison)
```
</details>

This summary is followed by the paired test-example analysis, seed analysis, confidence interval, category slices, and qualitative disagreement review. None of these should be replaced by a single p-value.

#### **Writing a Defensible Conclusion** {#writing-a-defensible-conclusion}

A weak conclusion overgeneralizes:

> Our method establishes a new state of the art for sentiment analysis and demonstrates that domain adaptation is universally beneficial.

The experiment does not support "sentiment analysis" across all domains and languages, nor universal benefit. It studies one benchmark, one adaptation corpus, one encoder family, and one protocol.

A defensible conclusion matches the evidence:

> On the locked ProductSentiment-1.0 group/temporal test split, continued masked-language pretraining on training-period product text improved mean macro-F1 from 0.817 to 0.829 across five matched fine-tuning seeds. The gain was accompanied by higher worst-category F1 (0.754 to 0.781) and nearly unchanged p95 inference latency (48 to 50 ms), but required an additional 36 GPU-hours of development compute. Under the preregistered paired evaluation, the uncertainty interval and randomization test supported a positive benchmark effect. The result is evidence that DAPT helps this product-review setting; it does not establish transfer to other domains, languages, encoders, or changing future data.

This conclusion contains:

- benchmark and split scope;
- intervention and strongest comparable baseline;
- mean effect across declared seeds;
- domain-relevant diagnostic evidence;
- deployment and development costs;
- statistical evidence without treating it as effect size;
- explicit boundaries on generalization.

The limitations should state that unlabeled DAPT data may overlap semantically with labeled product categories, the test period covers only one temporal shift, sentiment labels simplify mixed opinions, product-group splitting does not remove author-level correlation unless author IDs are available, and results may differ for multilingual or code-switched text.

Whether the candidate is called SOTA depends on the comparison set. If it exceeds every comparable published result under the same benchmark protocol, it may be leaderboard SOTA. Even without the top absolute score, the study can make a valuable causal-style claim about the contribution of DAPT. Scientific value does not require attaching SOTA to every improvement.

### **Practical Research Workflow** {#practical-research-workflow}

The complete workflow turns an idea into a claim through explicit gates:

$$
\text{question}
\rightarrow
\text{benchmark contract}
\rightarrow
\text{baseline ladder}
\rightarrow
\text{controlled development}
\rightarrow
\text{locked evaluation}
\rightarrow
\text{uncertainty and cost}
\rightarrow
\text{bounded claim}
\rightarrow
\text{reproducible artifacts}
$$

1. **Define the research question.** State the proposed mechanism and comparison. "Does domain-adaptive pretraining improve held-out product-category sentiment F1?" is testable; "Can we build a better model?" is not.

2. **Define the target population and independent unit.** Identify language, domain, period, user population, and whether the unit is a sentence, document, conversation, author, or group.

3. **Write the benchmark contract.** Lock dataset revision, split policy, adaptation rules, primary metric, diagnostics, evaluator, resource constraints, and prohibited information.

4. **Choose the minimum meaningful effect.** Decide what gain would matter scientifically or practically before seeing test results. This informs sample size, power, and interpretation.

5. **Build the baseline ladder.** Include data-independent, heuristic, classical, lightweight neural, and strong pretrained baselines as appropriate. Ensure the nearest baseline isolates the proposed change.

6. **Declare the development budget.** Specify search spaces, trial/compute limits, stopping rules, prompt-development policy, and how failed runs count.

7. **Develop only on training and validation evidence.** Conduct error analysis, ablations, and pilot tests without consulting final test labels or repeated leaderboard feedback.

8. **Freeze the final protocol.** Save split IDs, evaluator and prompt hashes, selected configurations, seed list, model revisions, and the planned statistical analysis.

9. **Run all planned experiments.** Generate immutable predictions and manifests for every baseline, candidate, and seed. Record failures instead of silently excluding them.

10. **Score centrally.** Use one evaluator, validate IDs, count malformed outputs, and produce aggregate plus diagnostic metrics from saved predictions.

11. **Quantify uncertainty and stability.** Separate test-sample uncertainty, training-seed variability, split variability, prompt variation, and human-rater variation according to the claim.

12. **Compare effect and cost.** Report absolute gain, interval, practical threshold, subgroup behavior, development compute, end-to-end inference cost, and Pareto status.

13. **Audit errors and threats to validity.** Inspect disagreements, annotation problems, shortcuts, contamination, domain shifts, and critical failure categories.

14. **Write a bounded conclusion.** Match every phrase to evidence and distinguish leaderboard, reproducible, practical, and Pareto SOTA.

15. **Release the evidence package.** Publish code, configuration, environment, artifact revisions, prediction files, manifests, seed-level results, and documented limitations when permitted.

The following configuration acts as a compact research contract. Values are illustrative and should be committed before final testing.

<details>
<summary>YAML: Reproducible NLP comparison contract</summary>

```yaml
project: product-sentiment-domain-adaptation
hypothesis:
  intervention: domain-adaptive masked-language pretraining
  primary_baseline: fine-tuned pretrained encoder
  expected_direction: candidate_macro_f1 > baseline_macro_f1

data:
  dataset: ProductSentiment
  version: "1.0"
  independent_unit: product_id
  split_policy: group_and_temporal
  locked_split_manifest: data/locked/splits.json
  contamination_audit: reports/train_test_overlap.json

evaluation:
  primary_metric: macro_f1
  diagnostics:
    - per_class_f1
    - worst_product_category_f1
    - expected_calibration_error
    - failed_output_rate
  evaluator_version: sentiment-eval-1.2.0
  minimum_meaningful_gain: 0.005
  independent_test_unit: product_id

development:
  search_regime: equal_gpu_hour_budget
  max_gpu_hours_per_neural_family: 24
  validation_only_selection: true
  failed_trials_count_toward_budget: true

randomness:
  training_seeds: [13, 21, 34, 55, 89]
  split_seed: 42
  final_test_repetitions: 1

statistics:
  primary_comparison: dapt_fine_tuned_encoder_vs_fine_tuned_encoder
  effect: absolute_macro_f1_difference
  interval: grouped_paired_bootstrap_95pct
  test: grouped_approximate_randomization_two_sided
  multiple_comparison_policy: holm_for_secondary_models

resources:
  inference_hardware: one_declared_gpu
  precision: fp16
  batch_size: 32
  report:
    - total_development_gpu_hours
    - p50_p95_p99_latency_ms
    - throughput_examples_per_second
    - peak_memory_gb

release:
  code_commit: required
  model_revision: required
  environment_lockfile: required
  raw_predictions: required_if_license_allows
  seed_level_scores: required
  limitations: required
```
</details>

Before accepting a result, apply three final gates:

| Gate | Pass condition | Action if it fails |
|---|---|---|
| Validity | Data, task, metric, and independent unit match the claim | Narrow or redesign the claim and benchmark |
| Reliability | Effect survives declared randomness and uncertainty analysis | Gather evidence or report an inconclusive result |
| Utility | Gain is meaningful relative to cost, risks, and intended use | Prefer a practical/Pareto alternative or revise constraints |

The workflow does not guarantee that the candidate wins. It guarantees that whichever conclusion emerges is interpretable. A strong baseline that defeats a complicated proposal is useful evidence; an uncertain difference is a valid result; a cheaper model on the Pareto frontier can be more valuable than a tiny absolute SOTA gain.

Baselines, benchmarks, and SOTA therefore form one research discipline. Baselines establish what must be beaten, benchmarks define how evidence is collected, fair comparisons isolate why results differ, and SOTA describes only the strongest conclusion supported inside those boundaries.

